In [1]:
# =====================================================================================
# Project: PTB-XL Metadata Audit  -  v0.8 FINAL TRAINING on folds 1-9 (18 runs, NO evaluation)
# Code version: v0.8
# Date: 2026-10-03
# Locked by: final_run_protocol.json (FRP) -> stage_plan["v0.8"], v0.8_overrides,
#            crash_and_rerun_rules["v0.8"], locked_quantities (FRP has priority over any prompt)
#
# Purpose:
#   STEP 0 (part 1)  snapshot PREREG_DIR (10 files); verify the five locked documents (self_sha256 +
#           file sha256) and their parent chain; the five .ots files (sha256 + size == v0.7 manifest);
#           abort if outputs\v0.9 exists; verify the v0.4 / v0.5 / v0.6 / v0.7 manifests; environment
#           (versions == v0.7 manifest, cuDNN 90100, same GPU); recipe FROM dev_protocol.json;
#           TRAINING_DETAILS rebuilt verbatim and asserted == v0.5_training_details.json["details"];
#           FRP stage_plan / v0.8_overrides / final_training_steps checks; input CSV and v0.4 cache
#           hashes; reference implementation sha256 (NOT exec'd)
#   STEP 0 (part 2)  folds 1-9 data, labels, physio columns; categorical encoder, age scaler and
#           per-lead normalization refit on folds 1-9 and asserted equal to FRP locked_quantities;
#           META matrices; the model uses the FRP locked lead-normalization values
#   PREFLIGHT  (CPU, before any file is written and before any attempt-log entry)
#           P1 init hashes of all 18 models (shared backbone/pool/pool_norm identical across arms)
#           P2 schedule preflight: float64 AdamW + OneCycleLR on a dummy parameter with zero grads,
#              7700 steps -> prod(1 - wd*lr) == FRP shrink factor, max lr == 0.01
#           P3 len(DataLoader) == 154 (built, not iterated)
#   TRAINING  18 runs (2 archs x 3 arms x 3 seeds), lr 0.01, 50 epochs, folds 1-9, append-only attempt
#           log, crash / re-run rule; per run: train-loss CSV + final state_dict (state_dict LAST)
#   MANIFEST  v0.8_training_summary.csv (no metric columns), v0.8_manifest.json
#
# BINDING FRP CLAUSES (verbatim meaning):
#   stage_plan["v0.8"]: 18 runs, lr 0.01, 50 epochs, everything else exactly as
#     v0.5_training_details.json with v0.8_overrides; encoder / age scaler / lead normalization = the
#     locked folds 1-9 values (asserted equal at v0.8 STEP 0); signal_only is EXPLORATORY;
#     NO evaluation of any kind; no fold-10 signal file opened, no fold-10 tensor built.
#   v0.8_overrides: train_data = folds 1-9 (19601 records); validation / evaluation / metric /
#     lr_tie_break / fold9_probs not applicable; OneCycle total_steps = 7700; no checkpoint selection,
#     the model after the last optimizer step is final.
#   crash_and_rerun_rules["v0.8"]: a run that crashes or yields a non-finite loss may be re-run from
#     scratch with the same seed BEFORE v0.9; if the re-run fails again: stop and report; once v0.9
#     has started, no re-run is allowed.
#
# EVALUATION: NONE. No metric of any kind is computed on any data (training data and fold 9 included;
# fold 9 is training data in v0.8). The only per-run quantity is the mean TRAINING loss
# (training_loss_logging). No AUROC function is defined, imported or called; no sklearn metric module is
# imported. The names of the development evaluation / metric helpers occur ONLY inside the locked
# TRAINING_DETAILS string literal (needed for its verbatim equality assert).
# FOLD 10: never opened; the CSV is parsed once and only strat_fold of fold-10 rows is used (count);
# signals only from the sha256-verified v0.4 folds 1-9 cache; wfdb is NOT imported (asserted).
# outputs\v0.7: only v0.7_manifest.json is read (hash-verified). No v0.5 / v0.7 run file is opened.
# PREREG_DIR: never written (snapshot asserted identical at the end).
#
# DOCUMENTED CHANGES vs v0.7 (all deliberate):
#   (1)  CODE_VERSION / OUT_DIR = v0.8; outputs\v0.9 must not exist (checked at start, before every run
#        and at the end).
#   (2)  Part A (LogisticRegression environment regression) REMOVED; replaced by STEP 0.6: VERSIONS ==
#        v0.7_manifest["versions"] (all keys except "os"; "os" mismatch = WARN), cuDNN == 90100, CUDA
#        available, GPU name == v0.7_manifest["gpu"]. LogisticRegression / metric / rankdata imports
#        removed; the AUROC helper, write_once_json, save_csv and every evaluation helper removed.
#   (3)  Part B (reused v0.5 runs), Part C (analysis plan), Part E (fold-9 analysis) REMOVED; the
#        reference implementation is sha256-verified only, NOT exec'd.
#   (4)  STEP 0 adds: PREREG_DIR has 10 files; .ots sha256 + size == v0.7_manifest["ots_files"]; v0.7
#        manifest (file sha256, flags, parent_documents); FRP earlier_manifests for v0.4/v0.5; FRP
#        stage_plan["v0.8"] phrases, v0.8_overrides total_steps 7700, locked final_training_steps.
#        The v0.7 stage_plan["v0.7"] checks and the dev_protocol selection_rule / sanity_gate text
#        asserts are dropped (fold-9 evaluation texts, not used in v0.8).
#   (5)  Data: the fold-10 row count (strat_fold only) is asserted == 2198, then the frame is restricted
#        to strat_fold <= 9 and the unrestricted frame is DELETED; training rows = ALL folds 1-9 rows.
#   (6)  Encoder: folds 1-9 refit (v0.6 ENC19 logic, train_mask all-True), asserted against FRP
#        categories / widths / new levels / column order / all-zero columns; folds 1-8 categories built
#        only for the new_levels assert. Age scaler refit on folds 1-9, asserted against FRP. The v0.7
#        folds 1-8 lead-norm and folds 1-8 age-scaler regression checks are dropped.
#   (7)  Lead normalization: lead_norm_two_pass (v0.6 verbatim) over all folds 1-9 rows, asserted atol
#        1e-12 against FRP; the model uses the FRP locked values (LEAD_MEAN_F32 / LEAD_STD_F32 globals
#        read by RandomCropDataset, verbatim from v0.7).
#   (8)  PREFLIGHT P1-P3 (new). Nothing (not even OUT_DIR) is written before STEP 0 and the preflight
#        pass. v0.8_preflight_init_hashes.csv is write-once: on re-execution it is recomputed and
#        compared row by row, never overwritten.
#   (9)  train_run: (a) fold-9 prediction / probability CSV / AUROC removed; (b) the model is constructed
#        on CPU and its init hashes are asserted equal to the preflight BEFORE .to(DEVICE) (the v0.7
#        one-liner ECGNet(...).to(DEVICE) is split in two; same seeding and construction order);
#        (c) asserts head width 256/259/339, len(loader) == 154, total steps 7700, last batch 17;
#        (d) lr read immediately before every opt.step(), prod *= (1 - wd*lr) in float64, asserted
#        == FRP shrink factor (tol 1e-9) with 7700 steps; (e) zero-column check for signal_plus_all
#        (WARN only); (f) every floating state_dict tensor asserted finite; (g) loss WARN vs the v0.7
#        median (WARN only); (h) save order: train-loss CSV, then state_dict LAST (both .partial ->
#        os.replace); then log "completed". "state_dict exists" <=> "run complete". The plan-file
#        assert of v0.7 is removed (no analysis plan in v0.8).
#   (10) Crash / re-run rule keyed on the state_dict (was: the probability CSV).
#   (11) Outputs: no probability file, no table with a metric column.
#
# INIT HASH RECIPE (init_shared_sha256 / init_full_sha256):
#   sha256 over the state_dict entries in sorted(key) order; for each entry the bytes
#   key.encode("utf-8") + b"\0" + str(t.dtype).encode("utf-8") + b"\0" + str(tuple(t.shape)).encode("utf-8")
#   + b"\0" + t.detach().cpu().contiguous().numpy().tobytes() + b"\0" are fed to ONE hashlib.sha256 object.
#   init_shared = keys starting with "backbone.", "pool." or "pool_norm." (BN buffers included);
#   init_full = all keys. Computed right after set_all_seeds(seed); ECGNet(arch, meta_dim) on CPU.
#
# RE-RUN BEHAVIOUR (crash_and_rerun_rules["v0.8"]; log = OUT_DIR\v0.8_attempt_log.jsonl, append-only):
#   - state_dict exists -> COMPLETE, never re-trained; both file hashes verified against the last
#     "completed" entry and its init hashes == preflight; if no such entry: the loss CSV must exist,
#     "completed_recovered" is logged with init/shrink fields null, WARN;
#   - no state_dict -> if >= 2 "started" entries: STOP (no setting may be changed except through a
#     written, externally timestamped amendment made before v0.9); else leftover files of the run are
#     deleted (logged) and the run is trained from scratch with the same seed as attempt n+1.
#   - every abnormal end consumes an attempt: exception (incl. non-finite loss / parameters, CUDA error),
#     KeyboardInterrupt, kernel death ("started" without "completed"). A sleep the kernel survives is not.
#   - the summary CSV and the manifest are regenerated on every execution; run files are never touched
#     once complete.
#
# Outputs: OUT_DIR = <PROJECT_ROOT>\outputs\v0.8
#   per run: v0.8_{arch}_{arm}_seed{seed}_lr0.01_train_loss.csv, v0.8_{arch}_{arm}_seed{seed}_lr0.01_state_dict.pt
#   v0.8_preflight_init_hashes.csv (write-once), v0.8_attempt_log.jsonl (append-only),
#   v0.8_training_summary.csv, v0.8_manifest.json (written last)
# =====================================================================================

import os, sys, json, ast, hashlib, platform, time, random, math, re, gc, ntpath
from datetime import datetime
import numpy as np
import pandas as pd
import scipy                                         # version assert only
import sklearn                                       # version assert only
from sklearn.preprocessing import StandardScaler     # age scaler

assert "wfdb" not in sys.modules, "wfdb is imported in this kernel - restart the kernel (v0.8 must not use wfdb)."
_WFDB_AT_START = "wfdb" in sys.modules

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ------------------------------- configuration ---------------------------------------
PROJECT_ROOT = r"D:\ptbxl-metadata-audit"
DATA_ROOT  = os.path.join(PROJECT_ROOT, "data", "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
OUT_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.8")
V04_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.4")
V05_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.5")
V06_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.6")
V07_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.7")
V09_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.9")
PREREG_DIR = os.path.join(PROJECT_ROOT, "preregistration")

SEAL_PATH             = os.path.join(PREREG_DIR, "pre_registration.json")
DEV_PROTOCOL_PATH     = os.path.join(PREREG_DIR, "dev_protocol.json")
TRAINING_DETAILS_PATH = os.path.join(PREREG_DIR, "v0.5_training_details.json")
LR_SELECTION_PATH     = os.path.join(PREREG_DIR, "lr_selection.json")
FINAL_PROTOCOL_PATH   = os.path.join(PREREG_DIR, "final_run_protocol.json")
DOC_PATHS = {"pre_registration": SEAL_PATH, "dev_protocol": DEV_PROTOCOL_PATH,
             "training_details": TRAINING_DETAILS_PATH, "lr_selection": LR_SELECTION_PATH,
             "final_run_protocol": FINAL_PROTOCOL_PATH}

V04_MANIFEST = os.path.join(V04_DIR, "v0.4_manifest.json")
V05_MANIFEST = os.path.join(V05_DIR, "v0.5_manifest.json")
V06_MANIFEST = os.path.join(V06_DIR, "v0.6_manifest.json")
V07_MANIFEST = os.path.join(V07_DIR, "v0.7_manifest.json")      # the ONLY file read from outputs\v0.7
SIGNALS_NPY = os.path.join(V04_DIR, "v0.4_signals_folds1to9_100hz.npy")
SIGNALS_INDEX = os.path.join(V04_DIR, "v0.4_signals_index.csv")
V04_LEAD_NORM_CSV = os.path.join(V04_DIR, "v0.4_lead_norm_params.csv")

CODE_VERSION = "v0.8"
PREFIX = CODE_VERSION + "_"
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
ACQ_COLS = ["device", "site", "nurse"]

# --- the five locked documents: self_sha256 (canonical JSON) and file sha256 (bytes on disk) ---
# >>> verbatim from v0.7 (document hash constants)
EXPECTED_SELF_SHA256 = {
    "pre_registration":   "5183f54569f3358a5339b16a2c3c3be49f5c7798eabe6b0bb25946119c2ba469",
    "dev_protocol":       "c967ac38c8cc9382bf41fdd3785c8bbd730db196e349846b8052d856bbac5fc6",
    "training_details":   "c12c33a2f649984fb1b70a46445823c46b57d1b024d87655e16661789f04b46f",
    "lr_selection":       "c70cbfa65e0ab75321dfef9c64714a64705c60a967685d79a878a1d40450749d",
    "final_run_protocol": "834acddec288862d0418e9ee70204898c9d371788eed7eb362f4c0d63d305c6b",
}
EXPECTED_FILE_SHA256 = {
    "pre_registration":   "11dd446272654fc80d65b2fdd40259fa674bbb419a44fde84e589b6e297cde32",
    "dev_protocol":       "babf101c1d69b322daef6907175fdc4c17b5a53755a68f13a998eefa0148dc2d",
    "training_details":   "6480c191aeaed9ff1e6b1da5caa0c1e46908d453d30106a32beb7cf83678be11",
    "lr_selection":       "358d783b37bba6fcf61237a43817c1fd8594fe192f8ecdfca2c25b871a21415b",
    "final_run_protocol": "e1a5260e72d2e8b0d27dc709639468b7f9a48a7c8e2f8af2d663e105dbd329d4",
}
OTS_NOTE = "Bitcoin attestation of all five .ots files confirmed by the user on 2026-10-02 (not re-verified from Python)"

EXPECTED_V04_MANIFEST_SHA256 = "3781f90c4cf5a85ea612efabb1fd81cc61b77cbeb38a54cc644c10c66afe1d9b"
EXPECTED_V05_MANIFEST_SHA256 = "59968bfb50d8c919de0e8004ef878a44a115560de438b7e74582877495e95f87"
EXPECTED_V06_MANIFEST_SHA256 = "2cb7a1b023e03aad85bb11150ad38e203624300170bc0fa197014c727758c61a"
EXPECTED_REFERENCE_CODE_SHA256 = "48ac837c07880f0bd88a7bd01475e7da3edbeab46019f8288976884ad7662856"

EXPECTED_INPUT_HASHES = {
    "ptbxl_database.csv": "7600de9c1b27d181d850b3c6038a35d7c3ddb6bb33b702e3a20252a6859d216b",
    "scp_statements.csv": "ad05b0b1fcae83bb1230755ad9cfc7c96f303feddc08a4a9ad5bdc9ca63bac8f",
}
EXPECTED_SIGNALS_NPY_SHA256 = "17558f9d09b39b990d7f3a5ca2ea10e1527980288a49a6828c89b7c3df61745a"
EXPECTED_SIGNALS_INDEX_SHA256 = "06b49ecdafc6a74c9bdbcfa0177bb357eb845ece187b40e3a777496a8cf72203"
EXPECTED_LEAD_NORM_CSV_SHA256 = "07fb6d6dc8f0f1e2379fdd45b6462f1102518621eb242ba585fb8600acff35aa"

EXPECTED_FOLD9_POSITIVES = {"NORM": 955, "MI": 540, "STTC": 528, "CD": 495, "HYP": 268}
EXPECTED_N_RECORDS_FOLDS1TO9 = 19601
# <<< verbatim from v0.7
EXPECTED_V07_MANIFEST_SHA256 = "b443264cb1096ebd35dec5340f27e31d2a4f09795165f1aaca27070c673c2dc6"
EXPECTED_V07_FLAGS = {"fold10_opened": False, "prereg_dir_unchanged": True, "h3_verdict_called": False}
EXPECTED_PREREG_N_FILES = 10
EXPECTED_N_FOLD10_RECORDS = 2198
EXPECTED_N_AGE_SENTINEL_FOLDS1TO9 = 259
EXPECTED_CUDNN_VERSION = 90100

# >>> verbatim from v0.7 (constants that dev_protocol.json defines; the FILE values are used)
EXPECTED_EPOCHS = 50
EXPECTED_LR_GRID = [1e-3, 3e-3, 1e-2]
EXPECTED_CROP = 250
EXPECTED_EVAL_STRIDE = 125
EXPECTED_WINDOW_STARTS = [0, 125, 250, 375, 500, 625, 750]
EXPECTED_WEIGHT_DECAY = 1e-2
EXPECTED_BATCH_SIZE = {"resnet1d_wang": 128, "inception1d": 128}
EXPECTED_SELECTED_LR = {"resnet1d_wang": 0.01, "inception1d": 0.01}

SEED = 42
FS = 100
N_SAMPLES = 1000
N_LEADS = 12
NUM_WORKERS = 0                   # Windows + Jupyter
# <<< verbatim from v0.7
# The next three constants exist ONLY because the locked TRAINING_DETAILS text (verbatim equality assert)
# contains them. Nothing is evaluated in v0.8.
EVAL_BATCH_SIZE = 512             # appears in TRAINING_DETAILS["evaluation"] text only
LR_TIE_TOL = 1e-4                 # appears in TRAINING_DETAILS["lr_tie_break"] text only
SKLEARN_CHECK_TOL = 1e-9          # appears in TRAINING_DETAILS["metric"] text only

ARCHS = ["resnet1d_wang", "inception1d"]
ARMS = ["signal_only", "signal_plus_physio", "signal_plus_all"]
SEEDS = [42, 123, 2024]
MAX_ATTEMPTS = 2
EMBEDDING_DIM = 256
CHUNK = 1000

# final-run (folds 1-9) locked quantities; every one is also asserted equal to FRP below
EXPECTED_ONEHOT_WIDTHS_FINAL = {"device": 13, "site": 53, "nurse": 14}
EXPECTED_NEW_LEVELS = {"device": [], "site": ["049"], "nurse": []}
EXPECTED_META_DIM = {"signal_only": 0, "signal_plus_physio": 3, "signal_plus_all": 83}
EXPECTED_HEAD_IN = {"signal_only": 256, "signal_plus_physio": 259, "signal_plus_all": 339}
EXPECTED_N_BATCHES_FINAL = 154
EXPECTED_LAST_BATCH_FINAL = 17
EXPECTED_TOTAL_STEPS_FINAL = 7700
EXPECTED_FINAL_TRAINING_STEPS = {"n_records": 19601, "batch_size": 128, "n_batches_per_epoch": 154,
                                 "last_batch_size": 17, "epochs": 50, "onecycle_total_steps": 7700}
STAGE_PLAN_V08_PHRASES = ["18 runs", "lr 0.01", "50 epochs", "NO evaluation of any kind", "builds no fold-10 tensor"]

AGE_SCALER_TOL = {"mean": 1e-12, "scale": 1e-12, "var": 1e-9}
LEAD_NORM_ATOL = 1e-12
SHRINK_TOL = 1e-9
SCHED_PREFLIGHT_REL_TOL = 1e-12
LR_MAX_TOL = 1e-12
ZERO_COL_REL_DEV_WARN = 1e-4
LOSS_WARN_HIGH_FACTOR = 2.0
LOSS_WARN_LOW_FACTOR = 0.5
# Medians of the 9 final-epoch train losses per arch in the PRINTED v0.7 runs table (v0.7 notebook output,
# "PART E" table). Hard-coded; no v0.7 run file is read.
V07_MEDIAN_FINAL_LOSS = {"resnet1d_wang": 0.1829, "inception1d": 0.1397}
LOSS_WARN_NOTE = ("Loss WARN is WARN-only: it has no consequence and is never a re-run criterion (the only re-run "
                  "criterion is the locked one: a crash or a non-finite loss / parameters).")
ZERO_COL_WARN_NOTE = ("Zero-column rel_dev WARN is WARN-only: it has no consequence and is never a re-run criterion. "
                      "Reason for the check: an all-zero training column gets zero gradient -> the Adam step is exactly 0 "
                      "-> only AdamW's decoupled weight decay acts, so W_final should equal prod(1 - wd*lr) * W_init.")
INIT_HASH_RECIPE = (
    "sha256 over the state_dict entries in sorted(key) order; per entry: key.encode('utf-8') + b'\\0' + "
    "str(t.dtype).encode('utf-8') + b'\\0' + str(tuple(t.shape)).encode('utf-8') + b'\\0' + "
    "t.detach().cpu().contiguous().numpy().tobytes() + b'\\0', all fed to one hashlib.sha256 object; "
    "init_shared = keys starting with 'backbone.', 'pool.' or 'pool_norm.' (BN buffers included); "
    "init_full = all keys; computed after set_all_seeds(seed); ECGNet(arch, meta_dim) on CPU")
SHARED_PREFIXES = ("backbone.", "pool.", "pool_norm.")

_root_abs = os.path.abspath(DATA_ROOT)
for _p in (OUT_DIR, PREREG_DIR):
    assert not os.path.abspath(_p).lower().startswith(_root_abs.lower()), \
        "output path is inside the dataset folder - aborting: " + _p

RUN_STARTED = datetime.now().astimezone().isoformat()
WARNINGS = []


# >>> verbatim from v0.7 (helpers 1/3)
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()
# <<< verbatim from v0.7


# >>> verbatim from v0.7 (helpers 2/3)
def hdr(t):
    print("\n" + "=" * 100)
    print(t)
    print("=" * 100)


def warn(msg):
    WARNINGS.append(msg)
    print("[WARN] " + msg)


def note(msg):
    print("[NOTE] " + msg)


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def to_level_string(series, missing="__MISSING__"):
    """Human-readable, zero-padded categorical level for numeric columns (e.g.
    '007' for site) so sorting and printed feature names stay legible."""
    if pd.api.types.is_numeric_dtype(series):
        nn = series.dropna()
        integral = bool(len(nn) == 0 or np.all(np.isclose(nn.values, np.round(nn.values))))
        if integral:
            width = max(3, len(str(int(nn.max()))) if len(nn) else 3)
            return series.apply(lambda v: missing if pd.isna(v) else str(int(v)).zfill(width))
    return series.apply(lambda v: missing if pd.isna(v) else str(v).strip())


def encode_categorical(level_series, train_mask, col_name):
    """One-hot encode a pre-stringified categorical column. The category list is
    fixed from the training rows only, always includes an explicit '__MISSING__'
    level, and always includes a catch-all '__UNKNOWN__' level for anything seen
    only outside training (e.g. only in fold 9). Both splits therefore produce the
    exact same set of columns."""
    train_vals = level_series[train_mask]
    train_categories = sorted(set(train_vals.unique().tolist()) | {"__MISSING__"})
    known = set(train_categories)
    remapped = level_series.where(level_series.isin(known), other="__UNKNOWN__")
    categories = train_categories + ["__UNKNOWN__"]
    cat = pd.Categorical(remapped, categories=categories)
    dummies = pd.get_dummies(cat)
    dummies.columns = ["{}={}".format(col_name, c) for c in dummies.columns]
    dummies.index = level_series.index
    return dummies, categories


def canonical_json_bytes(d):
    """Canonical serialization used for self_sha256 (see SELF_SHA256_PROCEDURE)."""
    return json.dumps(d, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                      default=_json_default).encode("utf-8")


SELF_SHA256_PROCEDURE = (
    "Load the JSON file; remove the key 'self_sha256' (the key must be absent, not empty); "
    "serialize the remaining dict with json.dumps(d, sort_keys=True, ensure_ascii=False, "
    "separators=(',', ':')); compute SHA-256 of the UTF-8 bytes; the hex digest must equal "
    "the stored 'self_sha256'. (At write time numpy scalars were converted to plain JSON "
    "types first, so the loaded file hashes identically.)"
)


def verify_self_sha256(path):
    """Return (stored, recomputed, loaded_dict) for a self-hashed JSON document."""
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    stored = d.pop("self_sha256", None)
    recomputed = hashlib.sha256(canonical_json_bytes(d)).hexdigest()
    d["self_sha256"] = stored
    return stored, recomputed, d
# <<< verbatim from v0.7


# >>> verbatim from v0.7 (helpers 3/3)
def plain(o):
    return json.loads(json.dumps(o, ensure_ascii=False, default=_json_default))


def manifest_entry(manifest, basename):
    """Return the unique 'outputs' entry of a project manifest whose file basename matches
    (works for full Windows paths and bare names)."""
    hits = [o for o in manifest["outputs"] if ntpath.basename(o["file"]) == basename]
    assert len(hits) == 1, "expected exactly one manifest entry for {}, found {}".format(basename, len(hits))
    return hits[0]


def load_verified_manifest(path, expected_sha, label):
    assert os.path.isfile(path), label + " not found: " + path + " - aborting (fix the path constant)."
    got = sha256_of(path)
    assert got == expected_sha, "{} file sha256 {} != expected {} - aborting.".format(label, got, expected_sha)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f), got


def prereg_snapshot():
    """Sorted (name, sha256) of every file directly in PREREG_DIR (incl. the .ots files)."""
    out = []
    for name in sorted(os.listdir(PREREG_DIR)):
        p = os.path.join(PREREG_DIR, name)
        if os.path.isfile(p):
            out.append((name, sha256_of(p)))
    return out
# <<< verbatim from v0.7


def assert_no_v09(where):
    """crash_and_rerun_rules['v0.8']: once v0.9 has started, no re-run is allowed."""
    assert not os.path.exists(V09_DIR), \
        "{} exists ({}) - v0.9 has started; no v0.8 (re-)run is allowed - aborting.".format(V09_DIR, where)


def save_csv_atomic(df, path):
    """Write a CSV to '<path>.partial' and rename it (no half-written file under the final name)."""
    df.to_csv(path + ".partial", index=False, encoding="utf-8-sig")
    os.replace(path + ".partial", path)
    return sha256_of(path)


def state_dict_sha256(sd, keys):
    """INIT_HASH_RECIPE (see header)."""
    h = hashlib.sha256()
    for k in sorted(keys):
        t = sd[k].detach().cpu().contiguous()
        h.update(k.encode("utf-8")); h.update(b"\0")
        h.update(str(t.dtype).encode("utf-8")); h.update(b"\0")
        h.update(str(tuple(t.shape)).encode("utf-8")); h.update(b"\0")
        h.update(t.numpy().tobytes()); h.update(b"\0")
    return h.hexdigest()


def init_hashes(model):
    sd = model.state_dict()
    shared = [k for k in sd if k.startswith(SHARED_PREFIXES)]
    assert shared and len(shared) < len(sd), "unexpected state_dict layout for the shared-init hash"
    return state_dict_sha256(sd, shared), state_dict_sha256(sd, list(sd.keys()))


# =====================================================================================
# STEP 0 (part 1) - INTEGRITY (nothing is written before STEP 0 and the preflight pass)
# =====================================================================================
hdr("STEP 0 (part 1) - INTEGRITY")

# --- 1. PREREG_DIR snapshot (asserted identical at the end of the cell) ---
# >>> verbatim from v0.7 (PREREG_DIR snapshot)
assert os.path.isdir(PREREG_DIR), "PREREG_DIR not found: " + PREREG_DIR
PREREG_SNAPSHOT_START = prereg_snapshot()
print("PREREG_DIR snapshot: {} files".format(len(PREREG_SNAPSHOT_START)))
for name, h in PREREG_SNAPSHOT_START:
    print("  {}  {}".format(h, name))
# <<< verbatim from v0.7
assert len(PREREG_SNAPSHOT_START) == EXPECTED_PREREG_N_FILES, \
    "PREREG_DIR has {} files, expected {} - aborting.".format(len(PREREG_SNAPSHOT_START), EXPECTED_PREREG_N_FILES)

# --- 2. the five locked documents ---
# >>> verbatim from v0.7 (five documents + parent chain)
DOCS, DOC_SELF, DOC_FILE = {}, {}, {}
for key, path in DOC_PATHS.items():
    assert os.path.isfile(path), key + " not found: " + path + " - aborting."
    _s, _r, _d = verify_self_sha256(path)
    assert _s == _r == EXPECTED_SELF_SHA256[key], \
        "{}: self_sha256 stored {} / recomputed {} / expected {} - aborting.".format(key, _s, _r, EXPECTED_SELF_SHA256[key])
    _f = sha256_of(path)
    assert _f == EXPECTED_FILE_SHA256[key], "{}: file sha256 {} != {} - aborting.".format(key, _f, EXPECTED_FILE_SHA256[key])
    DOCS[key], DOC_SELF[key], DOC_FILE[key] = _d, _s, _f
    print("{:18s} self {} | file {} -> PASS".format(key, _s, _f))
SEAL, DEV, TD_DOC, LR_SEL, FRP = (DOCS["pre_registration"], DOCS["dev_protocol"], DOCS["training_details"],
                                  DOCS["lr_selection"], DOCS["final_run_protocol"])
assert SEAL.get("sealed") is True
assert DEV["parent_seal_self_sha256"] == DOC_SELF["pre_registration"]
assert TD_DOC["parent_seal_self_sha256"] == DOC_SELF["pre_registration"]
assert TD_DOC["parent_dev_protocol_self_sha256"] == DOC_SELF["dev_protocol"]
assert LR_SEL["parent_hashes"] == {"pre_registration": DOC_SELF["pre_registration"],
                                   "dev_protocol": DOC_SELF["dev_protocol"],
                                   "training_details": DOC_SELF["training_details"]}
assert FRP["parent_hashes"] == {k: {"self_sha256": DOC_SELF[k], "file_sha256": DOC_FILE[k]}
                                for k in ("pre_registration", "dev_protocol", "training_details", "lr_selection")}, \
    "final_run_protocol parent_hashes differ from the four documents on disk - aborting."
assert FRP["code_version"] == "v0.6" and FRP["write_once"] is True
print("parent chain: dev_protocol -> seal; training_details -> seal + dev_protocol; lr_selection -> 3; "
      "final_run_protocol.parent_hashes == the four documents (self + file): PASS")
# <<< verbatim from v0.7

# --- 3. OpenTimestamps proofs: existence, sha256, size (compared with the v0.7 manifest in step 5) ---
# >>> verbatim from v0.7 (.ots files)
OTS_FILES = {}
for key, path in DOC_PATHS.items():
    ots = path + ".ots"
    assert os.path.isfile(ots), "OpenTimestamps proof missing: " + ots + " - aborting."
    OTS_FILES[os.path.basename(ots)] = {"sha256": sha256_of(ots), "bytes": os.path.getsize(ots)}
    print("  {}  {:6d} B  {}".format(OTS_FILES[os.path.basename(ots)]["sha256"], OTS_FILES[os.path.basename(ots)]["bytes"],
                                     os.path.basename(ots)))
print(OTS_NOTE)
# <<< verbatim from v0.7

# --- 4. v0.9 must not have started ---
assert_no_v09("STEP 0")
print("outputs\\v0.9 absent: PASS (no v0.9 run has started)")

# --- 5. earlier manifests (file sha256) ---
# >>> verbatim from v0.7 (v0.6 manifest)
V06_MAN, V06_MAN_SHA = load_verified_manifest(V06_MANIFEST, EXPECTED_V06_MANIFEST_SHA256, "v0.6_manifest.json")
assert V06_MAN["final_run_protocol"]["file_sha256"] == EXPECTED_FILE_SHA256["final_run_protocol"]
assert V06_MAN["final_run_protocol"]["self_sha256"] == EXPECTED_SELF_SHA256["final_run_protocol"]
print("v0.6_manifest.json    : {} | records final_run_protocol file sha256: PASS".format(V06_MAN_SHA))
# <<< verbatim from v0.7
V07_MAN, V07_MAN_SHA = load_verified_manifest(V07_MANIFEST, EXPECTED_V07_MANIFEST_SHA256, "v0.7_manifest.json")
assert V07_MAN["flags"] == EXPECTED_V07_FLAGS, "v0.7 manifest flags {} != {} - aborting.".format(V07_MAN["flags"], EXPECTED_V07_FLAGS)
assert V07_MAN["parent_documents"] == {k: {"self_sha256": DOC_SELF[k], "file_sha256": DOC_FILE[k]} for k in DOC_PATHS}, \
    "v0.7 manifest parent_documents differ from the five documents - aborting."
assert OTS_FILES == V07_MAN["ots_files"], "the .ots files (sha256 / size) differ from the v0.7 manifest - aborting."
print("v0.7_manifest.json    : {} | flags {} | parent_documents == the five documents | "
      ".ots sha256 + size == v0.7 manifest: PASS".format(V07_MAN_SHA, V07_MAN["flags"]))
V04_MAN, V04_MAN_SHA = load_verified_manifest(V04_MANIFEST, EXPECTED_V04_MANIFEST_SHA256, "v0.4_manifest.json")
V05_MAN, V05_MAN_SHA = load_verified_manifest(V05_MANIFEST, EXPECTED_V05_MANIFEST_SHA256, "v0.5_manifest.json")
assert FRP["earlier_manifests_sha256"]["v0.4_manifest.json"] == V04_MAN_SHA
assert FRP["earlier_manifests_sha256"]["v0.5_manifest.json"] == V05_MAN_SHA
V03_MAN_SHA_FROM_FRP = FRP["earlier_manifests_sha256"]["v0.3_manifest.json"]     # copied only; not opened
print("v0.4_manifest.json    : {} | v0.5_manifest.json: {} | both == final_run_protocol earlier_manifests: PASS".format(
    V04_MAN_SHA, V05_MAN_SHA))

# --- 6. environment (replaces v0.7 Part A) ---
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
VERSIONS = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "sklearn": sklearn.__version__, "torch": torch.__version__,
            "torch_cuda": torch.version.cuda, "os": platform.platform()}
print("versions:", VERSIONS)
_ver_diff = {k: (VERSIONS.get(k), V07_MAN["versions"].get(k)) for k in set(VERSIONS) | set(V07_MAN["versions"])
             if k != "os" and VERSIONS.get(k) != V07_MAN["versions"].get(k)}
assert not _ver_diff, "environment differs from v0.7 (now, v0.7): {} - aborting.".format(_ver_diff)
if VERSIONS["os"] != V07_MAN["versions"].get("os"):
    warn("os string differs from v0.7: now {!r}, v0.7 {!r}".format(VERSIONS["os"], V07_MAN["versions"].get("os")))
if not torch.cuda.is_available():
    raise RuntimeError("torch.cuda.is_available() is False - no usable CUDA GPU in this kernel. Aborting.")
CUDNN_VERSION = torch.backends.cudnn.version()
assert CUDNN_VERSION == EXPECTED_CUDNN_VERSION, "cuDNN {} != {} - aborting.".format(CUDNN_VERSION, EXPECTED_CUDNN_VERSION)
GPU_NAME = torch.cuda.get_device_name(0)
assert GPU_NAME == V07_MAN["gpu"], "GPU {!r} != v0.7 GPU {!r} - aborting.".format(GPU_NAME, V07_MAN["gpu"])
print("environment: versions == v0.7 manifest (except os) | cuDNN {} | GPU {} == v0.7: PASS".format(CUDNN_VERSION, GPU_NAME))

# --- 7. recipe values loaded FROM dev_protocol.json, then asserted equal to the constants ---
# >>> verbatim from v0.7 (recipe parsing)
_rec = DEV["training_recipe"]
EPOCHS = int(_rec["epochs"])
LR_GRID = [float(v) for v in DEV["tuning_grid"]["learning_rate"]]
WEIGHT_DECAY = float(_rec["weight_decay"])
BATCH_SIZE = {k: int(v) for k, v in _rec["batch_size"].items()}
_m = re.search(r"random (\d+)-sample crops", _rec["train_input"])
assert _m, "cannot parse the crop length from dev_protocol train_input"
CROP = int(_m.group(1))
_m = re.search(r"sliding windows of (\d+) samples, stride (\d+) \(window starts \[([0-9, ]+)\]\)", _rec["evaluation"])
assert _m, "cannot parse the evaluation windows from dev_protocol evaluation"
EVAL_WINDOW = int(_m.group(1))
EVAL_STRIDE = int(_m.group(2))
WINDOW_STARTS = [int(v) for v in _m.group(3).split(",")]
AGGREGATION = "mean" if "probability = mean over windows" in _rec["evaluation"] else None
assert "MEAN over sliding windows" in DEV["evaluation_aggregation_note"]
assert _rec["optimizer"] == "AdamW" and _rec["lr_schedule"] == "one-cycle"

assert EPOCHS == EXPECTED_EPOCHS, (EPOCHS, EXPECTED_EPOCHS)
assert LR_GRID == EXPECTED_LR_GRID, (LR_GRID, EXPECTED_LR_GRID)
assert WEIGHT_DECAY == EXPECTED_WEIGHT_DECAY, (WEIGHT_DECAY, EXPECTED_WEIGHT_DECAY)
assert BATCH_SIZE == EXPECTED_BATCH_SIZE, (BATCH_SIZE, EXPECTED_BATCH_SIZE)
assert CROP == EVAL_WINDOW == EXPECTED_CROP, (CROP, EVAL_WINDOW)
assert EVAL_STRIDE == EXPECTED_EVAL_STRIDE and WINDOW_STARTS == EXPECTED_WINDOW_STARTS
assert WINDOW_STARTS == list(range(0, N_SAMPLES - CROP + 1, EVAL_STRIDE))
assert AGGREGATION == "mean"
# <<< verbatim from v0.7
# (the evaluation-window values are parsed only because TRAINING_DETAILS' locked text contains them)
print("recipe from dev_protocol.json: epochs={} | weight_decay={} | batch sizes={} | crop={} -> all equal the "
      "constants: PASS".format(EPOCHS, WEIGHT_DECAY, BATCH_SIZE, CROP))

# >>> verbatim from v0.7 (selected lr)
SELECTED_LR = {k: float(v) for k, v in LR_SEL["selected_lr"].items()}
assert SELECTED_LR == EXPECTED_SELECTED_LR, "selected_lr {} != {} - aborting.".format(SELECTED_LR, EXPECTED_SELECTED_LR)
# <<< verbatim from v0.7
print("selected lr (lr_selection.json): {} -> PASS".format(SELECTED_LR))

# --- TRAINING_DETAILS rebuilt VERBATIM (v0.5 / v0.7) and asserted equal to the locked file ---
# NOTE: the "evaluation", "metric", "validation", "lr_tie_break" and "per_run_outputs" texts below describe the
# development stage and name the development evaluation / metric helpers; they exist ONLY so that this
# dict equals v0.5_training_details.json["details"] (verbatim equality assert). FRP v0.8_overrides replaces them
# for v0.8 (not applicable: validation, evaluation, metric, lr_tie_break, fold9_probs; train_data = folds 1-9).
# >>> verbatim from v0.7 (TRAINING_DETAILS)
ADAMW_BETAS = (0.9, 0.99)
ADAMW_EPS = 1e-8
ONECYCLE = {"pct_start": 0.3, "div_factor": 25.0, "final_div_factor": 1e4, "anneal_strategy": "cos",
            "cycle_momentum": True, "base_momentum": 0.85, "max_momentum": 0.95, "three_phase": False}

TRAINING_DETAILS = {
    "optimizer": {"name": "torch.optim.AdamW", "betas": list(ADAMW_BETAS), "eps": ADAMW_EPS,
                  "weight_decay": WEIGHT_DECAY, "weight_decay_source": "dev_protocol.json",
                  "applied_to": "all parameters (incl. BatchNorm and biases)"},
    "lr_scheduler": {"name": "torch.optim.lr_scheduler.OneCycleLR", "max_lr": "the run's lr",
                     "total_steps": "epochs * len(train_loader)", "stepped": "every batch (after optimizer.step())",
                     **ONECYCLE,
                     "momentum_note": ("cycle_momentum/base_momentum/max_momentum are torch defaults: AdamW "
                                       "beta1 is cycled 0.95 -> 0.85 -> 0.95 in the opposite phase of the lr")},
    "epochs": EPOCHS, "epochs_source": "dev_protocol.json",
    "batch_size": BATCH_SIZE, "batch_size_source": "dev_protocol.json",
    "train_data": ("folds 1-8; RandomCropDataset (v0.4): random {}-sample crop per record per epoch, start "
                   "uniform over [0, {}], per-lead normalized with folds 1-8 statistics".format(CROP, N_SAMPLES - CROP)),
    "data_order": ("DataLoader(shuffle=True, generator=torch.Generator().manual_seed(run_seed), num_workers=0, "
                   "pin_memory=True, drop_last=False); RandomCropDataset rng = np.random.default_rng(run_seed)"),
    "seeding": ("set_all_seeds(run_seed) (python random, numpy, torch, torch.cuda) immediately before model "
                "construction; cudnn.deterministic=True, cudnn.benchmark=False"),
    "training_loss": "BCEWithLogitsLoss (mean over batch and the 5 classes)",
    "training_loss_logging": ("per-epoch MEAN TRAINING loss = sample-weighted mean of batch losses over the "
                              "epoch (training data only), saved per run"),
    "validation": ("NO validation metric is computed at any epoch other than the end of epoch {}; no "
                   "checkpoint selection; the model after the last optimizer step is evaluated".format(EPOCHS)),
    "evaluation": ("model.eval(); torch.no_grad(); {} windows (starts {}) of {} samples per fold-9 record, "
                   "per-lead normalized with folds 1-8 statistics; per-window sigmoid PROBABILITIES averaged "
                   "per record (mean of probabilities, not of logits); metadata identical for all windows of a "
                   "record; inference batch size {} (no effect on results)".format(
                       len(WINDOW_STARTS), WINDOW_STARTS, CROP, EVAL_BATCH_SIZE)),
    "metric": ("macro-AUROC = mean over NORM, MI, STTC, CD, HYP of fast_auc (v0.2.1 function); verified once "
               "against sklearn.metrics.roc_auc_score (abs tol {})".format(SKLEARN_CHECK_TOL)),
    "lr_tie_break": ("highest fold-9 macro-AUROC; if two lrs are within {} of each other, choose the SMALLER lr "
                     "(implemented as: among lrs with macro-AUROC >= best - {}, take the smallest)".format(
                         LR_TIE_TOL, LR_TIE_TOL)),
    "per_run_outputs": ("v0.5_{arch}_{arm}_seed{seed}_lr{lr}_fold9_probs.csv (ecg_id, patient_id, 5 "
                        "probabilities), _train_loss.csv (per-epoch mean training loss), _state_dict.pt "
                        "(final model); all hashed in the manifest"),
    "determinism_note": ("torch.use_deterministic_algorithms is NOT enabled; some CUDA kernels may not be "
                         "bitwise reproducible across re-runs"),
}
assert plain(TRAINING_DETAILS) == TD_DOC["details"], \
    "v0.5_training_details.json on disk differs from this code's training definition - aborting."
TRAINING_DETAILS_SHA = DOC_SELF["training_details"]
# <<< verbatim from v0.7
print("TRAINING_DETAILS (this code) == v0.5_training_details.json['details']: PASS "
      "(FRP v0.8_overrides applied: folds 1-9 data / encoder / scaler / lead norm, total_steps 7700, no evaluation)")

# --- 8. FRP stage plan / overrides / locked step counts ---
STAGE_PLAN_V08 = FRP["stage_plan"]["v0.8"]
for _ph in STAGE_PLAN_V08_PHRASES:
    assert _ph in STAGE_PLAN_V08, "final_run_protocol stage_plan v0.8 does not contain {!r} - aborting.".format(_ph)
V08_OVERRIDES = FRP["v0.8_overrides"]
assert V08_OVERRIDES["lr_scheduler_total_steps_v0.8"] == EXPECTED_TOTAL_STEPS_FINAL
for _na in ("validation", "evaluation", "metric", "lr_tie_break"):
    assert _na in V08_OVERRIDES["not_applicable_in_v0.8"]
LQ = FRP["locked_quantities"]
assert LQ["final_training_steps"] == EXPECTED_FINAL_TRAINING_STEPS, \
    "locked final_training_steps {} != {} - aborting.".format(LQ["final_training_steps"], EXPECTED_FINAL_TRAINING_STEPS)
assert LQ["final_training_steps"]["epochs"] == EPOCHS
assert all(BATCH_SIZE[a] == LQ["final_training_steps"]["batch_size"] for a in ARCHS)
CRASH_RULE_V08 = FRP["crash_and_rerun_rules"]["v0.8"]
SHRINK_FINAL_EXACT = float(LQ["untrained_column_weight_shrink"]["final_folds1to9"]["shrink_factor_exact"])
assert int(LQ["untrained_column_weight_shrink"]["final_folds1to9"]["total_steps"]) == EXPECTED_TOTAL_STEPS_FINAL
print("final_run_protocol: stage_plan v0.8 phrases {} | v0.8_overrides total_steps {} | final_training_steps {} -> PASS".format(
    STAGE_PLAN_V08_PHRASES, V08_OVERRIDES["lr_scheduler_total_steps_v0.8"], LQ["final_training_steps"]))
print("crash rule v0.8 (locked): " + CRASH_RULE_V08)

# --- 9. input CSVs, v0.4 signal cache, lead-norm csv ---
# >>> verbatim from v0.7 (input CSVs + v0.4 cache)
DB_CSV = os.path.join(DATA_ROOT, "ptbxl_database.csv")
SCP_CSV = os.path.join(DATA_ROOT, "scp_statements.csv")
for p in (DB_CSV, SCP_CSV):
    assert os.path.isfile(p), "file not found: " + p
INPUT_HASHES = {os.path.basename(p): sha256_of(p) for p in (DB_CSV, SCP_CSV)}
assert INPUT_HASHES == EXPECTED_INPUT_HASHES == FRP["input_files_sha256"], "input CSV hashes differ - aborting."
print("input CSV hashes      : equal to v0.4-v0.6 and to final_run_protocol -> PASS")
V04_MAN, V04_MAN_SHA = load_verified_manifest(V04_MANIFEST, EXPECTED_V04_MANIFEST_SHA256, "v0.4_manifest.json")
CACHE_HASHES = {}
for path, expected in [(SIGNALS_NPY, EXPECTED_SIGNALS_NPY_SHA256), (SIGNALS_INDEX, EXPECTED_SIGNALS_INDEX_SHA256),
                       (V04_LEAD_NORM_CSV, EXPECTED_LEAD_NORM_CSV_SHA256)]:
    assert os.path.isfile(path), "v0.4 file missing: " + path
    got = sha256_of(path)
    assert got == expected == manifest_entry(V04_MAN, os.path.basename(path))["sha256"] \
        == FRP["v0.4_signal_cache_sha256"][os.path.basename(path)], "v0.4 file hash mismatch for " + path + " - aborting."
    CACHE_HASHES[os.path.basename(path)] = got
    print("  {}  {}  -> PASS (== v0.4 manifest == final_run_protocol)".format(got, os.path.basename(path)))
# <<< verbatim from v0.7

# --- 10. reference implementation: sha256 only (NOT exec'd in v0.8) ---
REFERENCE_CODE_SHA256 = hashlib.sha256(FRP["reference_implementation"]["code"].encode("utf-8")).hexdigest()
assert REFERENCE_CODE_SHA256 == FRP["reference_implementation"]["sha256_of_utf8_code"] == EXPECTED_REFERENCE_CODE_SHA256, \
    "reference implementation sha256 mismatch - aborting."
print("reference implementation: sha256 {} -> PASS (verified only; NOT exec'd in v0.8)".format(REFERENCE_CODE_SHA256))

# --- 11. existing v0.8 artefacts (re-execution) ---
ATTEMPT_LOG_PATH = os.path.join(OUT_DIR, PREFIX + "attempt_log.jsonl")
PREFLIGHT_CSV_PATH = os.path.join(OUT_DIR, PREFIX + "preflight_init_hashes.csv")
print("attempt log: {} | preflight CSV: {}".format(
    "present (re-execution)" if os.path.exists(ATTEMPT_LOG_PATH) else "not present",
    "present (will be compared, never overwritten)" if os.path.exists(PREFLIGHT_CSV_PATH) else "not present"))


# =====================================================================================
# STEP 0 (part 2) - DATA (folds 1-9), LABELS, LOCKED QUANTITIES (refit on folds 1-9, == FRP)
# =====================================================================================
hdr("STEP 0 (part 2) - DATA, LABELS, LOCKED QUANTITIES (folds 1-9)")

df_all = pd.read_csv(DB_CSV, index_col="ecg_id")
N_SOURCE = int(len(df_all))
n_fold10_in_source = int((df_all["strat_fold"] == 10).sum())       # strat_fold only, for a count
assert n_fold10_in_source == EXPECTED_N_FOLD10_RECORDS, \
    "fold-10 rows in source CSV {} != {} - aborting.".format(n_fold10_in_source, EXPECTED_N_FOLD10_RECORDS)
df9 = df_all[df_all["strat_fold"] <= 9].copy()
del df_all                                                          # no fold-10 value stays in memory
gc.collect()
FOLD10_ACCESS_LINE = (
    "fold-10 access: pd.read_csv parsed ptbxl_database.csv once (all {} rows, all columns); of the {} fold-10 rows "
    "only the strat_fold value was used, to count them; immediately afterwards the frame was restricted to "
    "strat_fold <= 9 and the unrestricted frame was deleted. No fold-10 signal file was opened, no fold-10 tensor "
    "was built, and no other fold-10 value is held in any variable or used in any computation.".format(
        N_SOURCE, n_fold10_in_source))
print(FOLD10_ACCESS_LINE)
assert set(df9["strat_fold"].unique()) == set(range(1, 10)), "working set is not exactly folds 1-9 - aborting."
assert (df9["strat_fold"] == 10).sum() == 0, "fold 10 leaked into the working set - aborting."
mask_train18 = df9["strat_fold"] <= 8          # folds 1-8: ONLY for the new_levels_vs_folds1to8 assert
mask_fold9 = df9["strat_fold"] == 9            # fold 9: ONLY for the fold-9 label COUNT assert (not a metric)
mask_final = df9["strat_fold"] <= 9            # folds 1-9: all True (final-run fit and training rows)
assert bool(mask_final.all())
# >>> verbatim from v0.7 (sidecar index frame)
current_index = pd.DataFrame({
    "ecg_id": df9.index.to_numpy(),
    "strat_fold": df9["strat_fold"].to_numpy(),
    "patient_id": df9["patient_id"].to_numpy(),
})
N_REC = len(current_index)
assert N_REC == EXPECTED_N_RECORDS_FOLDS1TO9
# <<< verbatim from v0.7
train_rows = np.arange(N_REC)                  # v0.8: ALL folds 1-9 rows are training rows
assert len(train_rows) == LQ["final_training_steps"]["n_records"]
print("records: folds 1-9 = {} (all training rows) | fold-10 rows in source CSV (counted, never used) = {}".format(
    N_REC, n_fold10_in_source))

# --- label derivation (verbatim from v0.2.1 via v0.7) ---
# >>> verbatim from v0.7 (label derivation)
scp = pd.read_csv(SCP_CSV, index_col=0)
diag = scp[scp["diagnostic"].fillna(0) == 1]
code2class = {k: v for k, v in diag["diagnostic_class"].items() if isinstance(v, str)}
parsed = df9["scp_codes"].apply(ast.literal_eval)
rec_classes = parsed.apply(lambda d: {code2class[k] for k in d if k in code2class})
lab = pd.DataFrame({c: rec_classes.apply(lambda s, c=c: int(c in s)) for c in SUPERCLASSES},
                   index=df9.index)
assert (current_index["ecg_id"].to_numpy() == lab.index.to_numpy()).all()
# <<< verbatim from v0.7
fold9_pos = {c: int(lab.loc[mask_fold9, c].sum()) for c in SUPERCLASSES}
assert fold9_pos == EXPECTED_FOLD9_POSITIVES, "fold-9 label counts differ from v0.2.1 - aborting."
print("fold-9 positives (label COUNT, not a metric):", fold9_pos, "-> PASS")
Y = lab[SUPERCLASSES].to_numpy(dtype=np.float32)
assert Y.shape == (N_REC, len(SUPERCLASSES))

# --- physio columns (verbatim from v0.4 via v0.7) ---
# >>> verbatim from v0.7 (physio columns)
age_raw = df9["age"]
age_sentinel_mask = (age_raw == 300)
n_age_sentinel = int(age_sentinel_mask.sum())
age_clean = age_raw.where(~age_sentinel_mask, np.nan)
age_censored = age_sentinel_mask.astype(float)
age_imputed = age_clean.fillna(90.0)
sex = df9["sex"].astype(float)
print("age: sentinel 300 affects", n_age_sentinel, "records; flagged via age_censored, imputed with 90.")
# <<< verbatim from v0.7
assert n_age_sentinel == EXPECTED_N_AGE_SENTINEL_FOLDS1TO9, \
    "age sentinel count {} != {} - aborting.".format(n_age_sentinel, EXPECTED_N_AGE_SENTINEL_FOLDS1TO9)
B1_cols = ["age_imputed", "age_censored", "sex"]
assert B1_cols == LQ["physio_columns"]["order"]

# --- categorical encoder refit on folds 1-9 (v0.6 ENC19 logic), asserted against FRP ---
LQ_ENC = LQ["categorical_encoder"]
# >>> verbatim from v0.6 (level strings + folds 1-9 encoder)
LEVELS = {col: to_level_string(df9[col]) for col in ACQ_COLS}
# <<< verbatim from v0.6
ENC19, CATS19, CATS18 = {}, {}, {}
for col in ACQ_COLS:
    ENC19[col], CATS19[col] = encode_categorical(LEVELS[col], mask_final, col)      # train_mask = all folds 1-9 rows
    _, CATS18[col] = encode_categorical(LEVELS[col], mask_train18, col)            # categories only (new-levels assert)
WIDTHS19 = {col: ENC19[col].shape[1] for col in ACQ_COLS}
NEW_LEVELS = {col: sorted(set(CATS19[col]) - set(CATS18[col])) for col in ACQ_COLS}
DROPPED_LEVELS = {col: sorted(set(CATS18[col]) - set(CATS19[col])) for col in ACQ_COLS}
del CATS18
for col in ACQ_COLS:
    assert CATS19[col] == LQ_ENC["categories"][col], \
        "{} categories differ from final_run_protocol (element by element) - aborting.".format(col)
assert WIDTHS19 == LQ_ENC["widths"] == EXPECTED_ONEHOT_WIDTHS_FINAL, "one-hot widths {} - aborting.".format(WIDTHS19)
assert NEW_LEVELS == LQ_ENC["new_levels_vs_folds1to8"] == EXPECTED_NEW_LEVELS, "new levels {} - aborting.".format(NEW_LEVELS)
assert all(len(v) == 0 for v in DROPPED_LEVELS.values()), "a folds 1-8 level vanished in folds 1-9 - aborting."
B3_COLS_FINAL = B1_cols + [c for col in ACQ_COLS for c in ENC19[col].columns]
assert B3_COLS_FINAL == LQ_ENC["signal_plus_all_column_order"], "signal_plus_all column order differs from FRP - aborting."
# >>> verbatim from v0.6 (all-zero training columns, folds 1-9 fit)
ALLZERO_TRAIN_COLS_19 = sorted(c for col in ACQ_COLS for c in ENC19[col].columns
                               if int(ENC19[col][c].sum()) == 0)
# <<< verbatim from v0.6
assert ALLZERO_TRAIN_COLS_19 == LQ_ENC["allzero_training_columns_folds1to9"], \
    "all-zero training columns {} != FRP - aborting.".format(ALLZERO_TRAIN_COLS_19)
print("encoder (folds 1-9): widths {} | new levels vs folds 1-8 {} | all-zero training columns {} | "
      "categories + column order == final_run_protocol: PASS".format(WIDTHS19, NEW_LEVELS, ALLZERO_TRAIN_COLS_19))

# --- age_imputed scaler refit on folds 1-9, asserted against FRP (values read FROM FRP) ---
LQ_AGE = LQ["age_scaler"]
age_scaler = StandardScaler().fit(age_imputed[mask_final].to_numpy().reshape(-1, 1))
AGE_SCALER_DIFFS = {"mean": abs(float(age_scaler.mean_[0]) - float(LQ_AGE["mean"])),
                    "scale": abs(float(age_scaler.scale_[0]) - float(LQ_AGE["scale"])),
                    "var": abs(float(age_scaler.var_[0]) - float(LQ_AGE["var"]))}
for _k, _tol in AGE_SCALER_TOL.items():
    assert AGE_SCALER_DIFFS[_k] <= _tol, "age scaler {}: |diff| {} > {} - aborting.".format(_k, AGE_SCALER_DIFFS[_k], _tol)
assert int(age_scaler.n_samples_seen_) == int(LQ_AGE["n_samples_seen"]) == N_REC
age_imputed_std = pd.Series(age_scaler.transform(age_imputed.to_numpy().reshape(-1, 1)).ravel(),
                            index=df9.index, name="age_imputed")
print("age_imputed scaler (folds 1-9): mean={!r} scale={!r} | max abs diff vs FRP {} | n_samples_seen {}: PASS".format(
    float(age_scaler.mean_[0]), float(age_scaler.scale_[0]), AGE_SCALER_DIFFS, int(age_scaler.n_samples_seen_)))

# --- META matrices (v0.7 construction with the folds 1-9 encoder) ---
META_FEATURES = pd.concat([age_imputed_std, age_censored.rename("age_censored"), sex.rename("sex"),
                           ENC19["device"], ENC19["site"], ENC19["nurse"]], axis=1)
ARM_COLUMNS = {"signal_only": [], "signal_plus_physio": B1_cols, "signal_plus_all": B3_COLS_FINAL}
assert EXPECTED_META_DIM == LQ["meta_dim"] and EXPECTED_HEAD_IN == LQ["head_first_linear_in_features"]
META = {}
for arm in ARMS:
    cols = ARM_COLUMNS[arm]
    if cols:
        META[arm] = META_FEATURES[cols].to_numpy(dtype=np.float32)
    else:
        META[arm] = np.zeros((N_REC, 0), dtype=np.float32)
    assert META[arm].dtype == np.float32 and META[arm].shape == (N_REC, EXPECTED_META_DIM[arm]), \
        "{}: metadata shape {} != {}".format(arm, META[arm].shape, (N_REC, EXPECTED_META_DIM[arm]))
    assert np.isfinite(META[arm]).all()
    assert EMBEDDING_DIM + EXPECTED_META_DIM[arm] == EXPECTED_HEAD_IN[arm]
    print("  {:20s} metadata dim = {:3d} | head first-Linear in_features {}".format(arm, META[arm].shape[1], EXPECTED_HEAD_IN[arm]))
assert list(META_FEATURES[ARM_COLUMNS["signal_plus_all"]].columns) == LQ_ENC["signal_plus_all_column_order"]
# indices (in hidden[0].weight columns) of the all-zero training columns, BY NAME from the locked column order
ZERO_COL_NAMES = list(LQ_ENC["allzero_training_columns_folds1to9"])
ZERO_COL_IDX = [EMBEDDING_DIM + LQ_ENC["signal_plus_all_column_order"].index(c) for c in ZERO_COL_NAMES]
assert all(float(np.abs(META["signal_plus_all"][:, i - EMBEDDING_DIM]).max()) == 0.0 for i in ZERO_COL_IDX)
print("all-zero columns {} -> hidden[0].weight column indices {} (by name, offset {})".format(
    ZERO_COL_NAMES, ZERO_COL_IDX, EMBEDDING_DIM))

# --- signal cache (v0.4), folds 1-9 only ---
# >>> verbatim from v0.7 (signal cache + sidecar index)
for path, expected in [(SIGNALS_NPY, EXPECTED_SIGNALS_NPY_SHA256), (SIGNALS_INDEX, EXPECTED_SIGNALS_INDEX_SHA256)]:
    assert os.path.isfile(path), "v0.4 cache file missing: " + path
    got = sha256_of(path)
    print("  {}  {}  -> {}".format(got, os.path.basename(path), "PASS" if got == expected else "FAIL"))
    assert got == expected, "v0.4 cache hash mismatch for " + path + " - aborting."
_old_idx = pd.read_csv(SIGNALS_INDEX, encoding="utf-8-sig")
assert list(_old_idx.columns) == list(current_index.columns) and _old_idx.astype(str).reset_index(drop=True).equals(
    current_index.astype(str).reset_index(drop=True)), "v0.4 sidecar index does not match folds 1-9 - aborting."
X = np.load(SIGNALS_NPY)
assert X.shape == (N_REC, N_LEADS, N_SAMPLES) and X.dtype == np.float32
print("signals loaded from the v0.4 cache: {} {} (folds 1-9 only)".format(X.shape, X.dtype))
# <<< verbatim from v0.7


# --- per-lead normalization refit on folds 1-9 (v0.6 code), asserted against FRP; FRP values are USED ---
# >>> verbatim from v0.6 (lead_norm_two_pass)
def lead_norm_two_pass(X, rows, chunk=CHUNK):
    """Per-lead mean / population std over the given rows; float64 two-pass in chunks of
    `chunk` records in row order (v0.4 Part B code, unchanged)."""
    _sum = np.zeros(N_LEADS, dtype=np.float64)
    for s in range(0, len(rows), chunk):
        _sum += X[rows[s:s + chunk]].astype(np.float64).sum(axis=(0, 2))
    _n_vals = len(rows) * N_SAMPLES
    mean = _sum / _n_vals
    _sq = np.zeros(N_LEADS, dtype=np.float64)
    for s in range(0, len(rows), chunk):
        xc = X[rows[s:s + chunk]].astype(np.float64) - mean[None, :, None]
        _sq += (xc ** 2).sum(axis=(0, 2))
    std = np.sqrt(_sq / _n_vals)
    return mean, std
# <<< verbatim from v0.6


LQ_LN = LQ["lead_normalization"]
t0 = time.time()
LEAD_MEAN_19, LEAD_STD_19 = lead_norm_two_pass(X, train_rows)
FRP_LEAD_MEAN = np.asarray(LQ_LN["mean"], dtype=np.float64)
FRP_LEAD_STD = np.asarray(LQ_LN["std"], dtype=np.float64)
LEAD_NORM_MAX_ABS_DIFF = {"mean": float(np.max(np.abs(LEAD_MEAN_19 - FRP_LEAD_MEAN))),
                          "std": float(np.max(np.abs(LEAD_STD_19 - FRP_LEAD_STD)))}
assert np.allclose(LEAD_MEAN_19, FRP_LEAD_MEAN, rtol=0, atol=LEAD_NORM_ATOL) and \
    np.allclose(LEAD_STD_19, FRP_LEAD_STD, rtol=0, atol=LEAD_NORM_ATOL), \
    "folds 1-9 lead normalization differs from final_run_protocol (atol {}): {} - aborting.".format(
        LEAD_NORM_ATOL, LEAD_NORM_MAX_ABS_DIFF)
_v04_lead_names = pd.read_csv(V04_LEAD_NORM_CSV, encoding="utf-8-sig")["lead"].tolist()   # lead order only
assert LQ_LN["lead_order"] == _v04_lead_names and len(_v04_lead_names) == N_LEADS, "lead order differs - aborting."
assert np.all(FRP_LEAD_STD > 0) and np.all(np.isfinite(FRP_LEAD_MEAN))
# the model uses the FRP LOCKED values (RandomCropDataset reads these two globals)
LEAD_MEAN_F32 = np.asarray(LQ_LN["mean"], dtype=np.float64).astype(np.float32)[:, None]
LEAD_STD_F32 = np.asarray(LQ_LN["std"], dtype=np.float64).astype(np.float32)[:, None]
print("per-lead normalization (folds 1-9, {} records, {:.1f}s) == final_run_protocol (atol {}; max abs diff {}) | "
      "lead order == FRP == v0.4 csv: PASS | the model uses the FRP locked values".format(
          N_REC, time.time() - t0, LEAD_NORM_ATOL, LEAD_NORM_MAX_ABS_DIFF))

LOCKED_CHECKS = {
    "categorical_encoder": {"categories_equal_frp": True, "widths": WIDTHS19, "new_levels_vs_folds1to8": NEW_LEVELS,
                            "signal_plus_all_column_order_equal_frp": True,
                            "allzero_training_columns_folds1to9": ALLZERO_TRAIN_COLS_19},
    "meta_dim": {arm: int(META[arm].shape[1]) for arm in ARMS},
    "head_first_linear_in_features_v0.8": EXPECTED_HEAD_IN,
    "age_scaler": {"mean": float(age_scaler.mean_[0]), "scale": float(age_scaler.scale_[0]),
                   "var": float(age_scaler.var_[0]), "n_samples_seen": int(age_scaler.n_samples_seen_),
                   "max_abs_diff_vs_frp": AGE_SCALER_DIFFS, "tolerances": AGE_SCALER_TOL},
    "lead_normalization": {"max_abs_diff_vs_frp": LEAD_NORM_MAX_ABS_DIFF, "atol": LEAD_NORM_ATOL,
                           "values_used": "FRP locked_quantities.lead_normalization (float64 -> float32)"},
    "final_training_steps": LQ["final_training_steps"],
    "fold9_positives_label_count": fold9_pos,
    "age_sentinel_count": n_age_sentinel,
    "fold10_rows_counted_in_source_csv": n_fold10_in_source,
    "zero_column_indices_in_hidden0": dict(zip(ZERO_COL_NAMES, ZERO_COL_IDX)),
}


# =====================================================================================
# GPU SETUP, MODELS (verbatim from v0.7), DATASET
# =====================================================================================
hdr("GPU SETUP AND MODELS")
DEVICE = torch.device("cuda:0")


# >>> verbatim from v0.7 (seeding + cuDNN flags)
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_all_seeds(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# <<< verbatim from v0.7
print("gpu: {} | torch {} | CUDA {} | cuDNN {} | cudnn.deterministic=True, benchmark=False".format(
    GPU_NAME, torch.__version__, torch.version.cuda, CUDNN_VERSION))


# >>> verbatim from v0.7 (MODELS)
# >>> MODELS
# SOURCE FOLLOWED: Strodthoff, Wagner, Schaeffter, Samek (2021) "Deep Learning for ECG
# Analysis: Benchmarks and Insights from PTB-XL", reference code repository
# helme/ecg_ptbxl_benchmarking: code/models/resnet1d.py (resnet1d_wang, BasicBlock1d,
# ResNet1d), code/models/inception1d.py (Inception1d, InceptionBlock1d, Shortcut1d),
# code/models/basic_conv1d.py (create_head1d, bn_drop_lin, AdaptiveConcatPool1d), and
# code/models/fastai_model.py (how the benchmark instantiates them). Re-implemented here
# from memory of that code in plain PyTorch (no fastai); the repository was NOT
# consulted at write time.
#
# DETAILS I AM UNSURE OF (should be checked against the repository):
#  1) resnet1d_wang kernel sizes: factory default kernel_size=[5,3], stem kernel 7,
#     stride_stem=1, no stem pooling, inplanes=128, layers [1,1,1], fix_feature_dim=True
#     (all stages 128 channels), stride 2 in stages 2 and 3. If fastai_model.py passes
#     the int kernel_size=5, BasicBlock1d expands it to [5, 5//2+1] = [5,3] -> same.
#  2) inception1d as used by the benchmark: kernel_size = 8*5 = 40 -> branch kernels
#     [39,19,9], depth 6, bottleneck 32, nb_filters 32 (output 4*32 = 128 channels),
#     residual shortcut every 3 blocks; all convs bias=False.
#  3) Head (create_head1d) with lin_ftrs_head=[128], ps_head=0.5, bn=True, relu,
#     concat pooling: AdaptiveConcatPool1d [max, avg] -> 2*nf -> BN, Dropout(0.25),
#     Linear(2*nf,128), ReLU -> BN, Dropout(0.5), Linear(128, 5). I believe the
#     benchmark default is lin_ftrs_head=[128]; with lin_ftrs_head=None the head would be
#     a single Linear(2*nf, 5) instead.
#  4) Order inside AdaptiveConcatPool1d is [max, avg] (fastai convention).
#  5) The reference aggregates sliding-window predictions with aggregate_fn="max" by
#     default in fastai_model.py (I am fairly but not fully sure); the project's dev
#     protocol uses the MEAN over windows (Part F). Flagged, not changed.
#  6) MaxPool1d(3, stride, padding=1) on the inception max-pool branch.
#
# METADATA FUSION (v0.4 PATCH FIX 1 -- sealed text is binding: "concatenated to the
# network's global-pooled embedding before the final linear classification head").
# The global-pooled embedding is the 2*nf = 256-dim AdaptiveConcatPool1d output; the
# "head" is the whole MLP head. The head's first block keeps the reference order
# (BN, Dropout(0.25), Linear, ReLU), split as: BN + Dropout applied to the embedding
# (`pool_norm`, identical across arms), then the RAW metadata (no BN, no dropout) is
# concatenated, then Linear(256 + meta_dim, 128) -> ReLU -> BN -> Dropout(0.5) ->
# Linear(128, 5). Backbone, pooling and pool_norm are identical across arms; the head's
# first Linear differs in input width by design.


def conv1d_same(in_planes, out_planes, kernel_size=3, stride=1):
    return nn.Conv1d(in_planes, out_planes, kernel_size=kernel_size, stride=stride,
                     padding=(kernel_size - 1) // 2, bias=False)


class BasicBlock1d(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, kernel_size=(3, 3), downsample=None):
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = [kernel_size, kernel_size // 2 + 1]
        self.conv1 = conv1d_same(inplanes, planes, kernel_size=kernel_size[0], stride=stride)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv1d_same(planes, planes, kernel_size=kernel_size[1])
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + residual)


def resnet1d_wang_backbone(input_channels=12, inplanes=128, kernel_size=(5, 3), kernel_size_stem=7,
                           stride_stem=1, layers=(1, 1, 1), stride=2):
    mods = [nn.Conv1d(input_channels, inplanes, kernel_size=kernel_size_stem, stride=stride_stem,
                      padding=(kernel_size_stem - 1) // 2, bias=False),
            nn.BatchNorm1d(inplanes), nn.ReLU(inplace=True)]          # no stem pooling
    cur = inplanes
    for i, n_blocks in enumerate(layers):
        st = 1 if i == 0 else stride
        planes = inplanes                                             # fix_feature_dim=True
        downsample = None
        if st != 1 or cur != planes:
            downsample = nn.Sequential(nn.Conv1d(cur, planes, kernel_size=1, stride=st, bias=False),
                                       nn.BatchNorm1d(planes))
        blocks = [BasicBlock1d(cur, planes, st, list(kernel_size), downsample)]
        cur = planes
        for _ in range(1, n_blocks):
            blocks.append(BasicBlock1d(cur, planes))
        mods.append(nn.Sequential(*blocks))
    return nn.Sequential(*mods), cur


class InceptionBlock1d(nn.Module):
    def __init__(self, ni, nb_filters, kss, stride=1, bottleneck_size=32):
        super().__init__()
        self.bottleneck = conv1d_same(ni, bottleneck_size, 1, stride) if bottleneck_size > 0 else nn.Identity()
        bi = bottleneck_size if bottleneck_size > 0 else ni
        self.convs = nn.ModuleList([conv1d_same(bi, nb_filters, ks) for ks in kss])
        self.conv_bottle = nn.Sequential(nn.MaxPool1d(3, stride, padding=1), conv1d_same(ni, nb_filters, 1))
        self.bn_relu = nn.Sequential(nn.BatchNorm1d((len(kss) + 1) * nb_filters), nn.ReLU())

    def forward(self, x):
        b = self.bottleneck(x)
        return self.bn_relu(torch.cat([c(b) for c in self.convs] + [self.conv_bottle(x)], dim=1))


class Shortcut1d(nn.Module):
    def __init__(self, ni, nf):
        super().__init__()
        self.act_fn = nn.ReLU(True)
        self.conv = conv1d_same(ni, nf, 1)
        self.bn = nn.BatchNorm1d(nf)

    def forward(self, inp, out):
        return self.act_fn(out + self.bn(self.conv(inp)))


class InceptionBackbone(nn.Module):
    def __init__(self, input_channels=12, kernel_size=40, depth=6, bottleneck_size=32, nb_filters=32,
                 use_residual=True):
        super().__init__()
        assert kernel_size >= 40 and depth % 3 == 0
        kss = [k - 1 if k % 2 == 0 else k for k in [kernel_size, kernel_size // 2, kernel_size // 4]]
        n_ks = len(kss) + 1
        self.depth, self.use_residual = depth, use_residual
        self.im = nn.ModuleList([InceptionBlock1d(input_channels if d == 0 else n_ks * nb_filters,
                                                  nb_filters, kss, bottleneck_size=bottleneck_size)
                                 for d in range(depth)])
        self.sk = nn.ModuleList([Shortcut1d(input_channels if d == 0 else n_ks * nb_filters, n_ks * nb_filters)
                                 for d in range(depth // 3)])
        self.out_channels = n_ks * nb_filters
        self.kss = kss

    def forward(self, x):
        input_res = x
        for d in range(self.depth):
            x = self.im[d](x)
            if self.use_residual and d % 3 == 2:
                x = self.sk[d // 3](input_res, x)
                input_res = x
        return x


class AdaptiveConcatPool1d(nn.Module):
    def __init__(self):
        super().__init__()
        self.mp = nn.AdaptiveMaxPool1d(1)
        self.ap = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        return torch.cat([self.mp(x), self.ap(x)], dim=1).flatten(1)


class ECGNet(nn.Module):
    """Backbone + reference head. Metadata (meta_dim >= 0) is concatenated, raw, to the
    global-pooled embedding (after its BN + Dropout) and enters the head's first Linear.
    backbone / pool / pool_norm are built in the same order with the same shapes for
    every arm, so with the same seed they are initialised identically (checked below)."""

    def __init__(self, arch, meta_dim, n_classes=5, input_channels=12, lin_ftrs=128, ps_head=0.5):
        super().__init__()
        if arch == "resnet1d_wang":
            self.backbone, nf = resnet1d_wang_backbone(input_channels=input_channels)
        elif arch == "inception1d":
            self.backbone = InceptionBackbone(input_channels=input_channels, kernel_size=40)
            nf = self.backbone.out_channels
        else:
            raise ValueError(arch)
        self.pool = AdaptiveConcatPool1d()
        # embedding pre-processing: identical across arms
        self.pool_norm = nn.Sequential(nn.BatchNorm1d(2 * nf), nn.Dropout(ps_head / 2))
        self.embedding_dim = 2 * nf
        self.meta_dim = meta_dim
        # head: receives [embedding (+ raw metadata)]
        self.hidden = nn.Sequential(nn.Linear(2 * nf + meta_dim, lin_ftrs), nn.ReLU(inplace=True),
                                    nn.BatchNorm1d(lin_ftrs), nn.Dropout(ps_head))
        self.final = nn.Linear(lin_ftrs, n_classes)

    def forward(self, x, meta=None):
        z = self.pool_norm(self.pool(self.backbone(x)))
        if self.meta_dim > 0:
            z = torch.cat([z, meta], dim=1)   # raw metadata: no BN, no dropout
        return self.final(self.hidden(z))
# <<< MODELS
# <<< verbatim from v0.7


# NOTE (v0.8): the "256 / 259 / 338" widths in this locked text are the folds 1-8 development widths; for v0.8
# final_run_protocol encoder_width_note governs (256 / 259 / 339). The text is asserted against dev_protocol.json only.
# >>> verbatim from v0.7 (METADATA_FUSION_DESCRIPTION)
METADATA_FUSION_DESCRIPTION = (
    "location: the metadata vector is concatenated to the network's global-pooled embedding (the "
    "2*nf = 256-dim AdaptiveConcatPool1d [max, avg] output, after that embedding's BatchNorm1d + "
    "Dropout(0.25)) and the concatenation is the input of the head's first Linear "
    "(256 + meta_dim -> 128), followed by ReLU, BatchNorm1d(128), Dropout(0.5), Linear(128 -> 5); "
    "metadata enters RAW (age_imputed standardized with training-fold statistics, all other columns "
    "0/1): no BatchNorm and no dropout is applied to the metadata; identical across arms: backbone, "
    "global pooling and the embedding's BatchNorm + Dropout (same seed -> identical initial weights, "
    "asserted in v0.4); differs across arms by design: only the input width of the head's first "
    "Linear (256 / 259 / 338 for signal_only / signal_plus_physio / signal_plus_all)")
assert METADATA_FUSION_DESCRIPTION == DEV["metadata_fusion"], \
    "ECGNet fusion description differs from the one locked in dev_protocol.json - aborting."
print("ECGNet metadata fusion == dev_protocol.json 'metadata_fusion': PASS")
# <<< verbatim from v0.7
assert "256 / 259 / 339" in FRP["encoder_width_note"]
print("head first-Linear widths for v0.8 (final_run_protocol encoder_width_note): {}".format(EXPECTED_HEAD_IN))


# NOTE (v0.8): the docstring is verbatim from v0.7; in v0.8 the globals LEAD_MEAN_F32 / LEAD_STD_F32 hold the
# FRP locked folds 1-9 values (set above), not folds 1-8 values.
# >>> verbatim from v0.7 (RandomCropDataset)
class RandomCropDataset(Dataset):
    """Training records -> random 250-sample crop, per-lead normalized with folds 1-8 stats."""

    def __init__(self, X, rows, meta, y, crop, seed):
        self.X, self.rows, self.meta, self.y, self.crop = X, rows, meta, y, crop
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        s = int(self.rng.integers(0, N_SAMPLES - self.crop + 1))
        x = (self.X[r, :, s:s + self.crop] - LEAD_MEAN_F32) / LEAD_STD_F32
        return (torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)),
                torch.from_numpy(self.meta[r]), torch.from_numpy(self.y[r]))
# <<< verbatim from v0.7


def make_loader(arch, arm, seed):
    """v0.7 DataLoader construction (verbatim arguments), folds 1-9 training rows."""
    ds = RandomCropDataset(X, train_rows, META[arm], Y, CROP, seed)
    return DataLoader(ds, batch_size=BATCH_SIZE[arch], shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                      drop_last=False, generator=torch.Generator().manual_seed(seed))


# =====================================================================================
# PREFLIGHT (CPU; before any file is written and before any attempt-log entry)
# =====================================================================================
hdr("PREFLIGHT (CPU)")
RUN_ORDER = [(arch, arm, seed) for arch in ARCHS for seed in SEEDS for arm in ARMS]      # as v0.7
assert len(RUN_ORDER) == len(ARCHS) * len(ARMS) * len(SEEDS) == 18

# --- P1 init hashes of all 18 models (same seeding + construction order as train_run, before .to(DEVICE)) ---
t0 = time.time()
PREFLIGHT = {}
preflight_rows = []
for arch, arm, seed in RUN_ORDER:
    set_all_seeds(seed)
    _model = ECGNet(arch, META[arm].shape[1])
    assert _model.hidden[0].in_features == EXPECTED_HEAD_IN[arm], \
        "{} {}: head first-Linear in_features {} != {}".format(arch, arm, _model.hidden[0].in_features, EXPECTED_HEAD_IN[arm])
    _sh, _fu = init_hashes(_model)
    PREFLIGHT[(arch, arm, seed)] = {"init_shared_sha256": _sh, "init_full_sha256": _fu}
    preflight_rows.append({"arch": arch, "arm": arm, "seed": seed, "meta_dim": int(META[arm].shape[1]),
                           "head_in_features": int(_model.hidden[0].in_features),
                           "init_shared_sha256": _sh, "init_full_sha256": _fu})
    del _model
gc.collect()
for arch in ARCHS:
    for seed in SEEDS:
        _shared = {PREFLIGHT[(arch, arm, seed)]["init_shared_sha256"] for arm in ARMS}
        assert len(_shared) == 1, "{} seed {}: shared backbone/pool/pool_norm init differs across arms - aborting.".format(arch, seed)
    assert len({PREFLIGHT[(arch, ARMS[0], s)]["init_shared_sha256"] for s in SEEDS}) == len(SEEDS), \
        "{}: different seeds gave the same shared init - aborting.".format(arch)
assert len({PREFLIGHT[k]["init_full_sha256"] for k in PREFLIGHT}) == len(PREFLIGHT)
PREFLIGHT_DF = pd.DataFrame(preflight_rows)
print("P1 init hashes: 18 models built on CPU in {:.1f}s | init_shared_sha256 identical across the 3 arms for every "
      "(arch, seed): PASS | distinct across seeds: PASS".format(time.time() - t0))

# --- P2 schedule preflight (float64, dummy parameter, zero grads -> only decoupled weight decay acts) ---
t0 = time.time()
_n_total = EPOCHS * EXPECTED_N_BATCHES_FINAL
_p = torch.nn.Parameter(torch.randn(16, dtype=torch.float64, generator=torch.Generator().manual_seed(SEED)))
_p_init = _p.detach().clone()
_lr = SELECTED_LR["resnet1d_wang"]
assert all(v == _lr for v in SELECTED_LR.values())
_opt = torch.optim.AdamW([_p], lr=_lr, betas=ADAMW_BETAS, eps=ADAMW_EPS, weight_decay=WEIGHT_DECAY)
_sched = torch.optim.lr_scheduler.OneCycleLR(_opt, max_lr=_lr, total_steps=_n_total, **ONECYCLE)
_prod, _steps, _lr_max = 1.0, 0, -1.0
for _ in range(_n_total):
    _p.grad = torch.zeros_like(_p)            # zeros, NOT None (AdamW skips grad=None params, weight decay included)
    _lr_now = float(_opt.param_groups[0]["lr"])
    _prod *= (1.0 - WEIGHT_DECAY * _lr_now)
    _lr_max = max(_lr_max, _lr_now)
    _steps += 1
    _opt.step()
    _sched.step()
_ref = _prod * _p_init
_rel = float((_p.detach() - _ref).abs().max() / _ref.abs().max())
SCHEDULE_PREFLIGHT = {"n_steps": _steps, "lr_product": _prod, "frp_shrink_factor_exact": SHRINK_FINAL_EXACT,
                      "abs_diff_vs_frp": abs(_prod - SHRINK_FINAL_EXACT), "tolerance": SHRINK_TOL,
                      "param_rel_dev_float64": _rel, "param_rel_dev_tolerance": SCHED_PREFLIGHT_REL_TOL,
                      "max_lr_seen": _lr_max, "max_lr_expected": _lr, "procedure": (
                          "CPU float64 dummy Parameter of 16 values (torch.randn, Generator seed {}); AdamW + OneCycleLR "
                          "exactly as train_run (total_steps = EPOCHS * {}); each step: grad = zeros, lr read before "
                          "opt.step(), prod *= 1 - wd*lr, opt.step(), sched.step()").format(SEED, EXPECTED_N_BATCHES_FINAL)}
assert _steps == EXPECTED_TOTAL_STEPS_FINAL, "schedule preflight steps {} != {} - aborting.".format(_steps, EXPECTED_TOTAL_STEPS_FINAL)
assert abs(_prod - SHRINK_FINAL_EXACT) <= SHRINK_TOL, \
    "schedule preflight prod {!r} != FRP shrink factor {!r} (tol {}) - aborting.".format(_prod, SHRINK_FINAL_EXACT, SHRINK_TOL)
assert _rel <= SCHED_PREFLIGHT_REL_TOL, "schedule preflight: W_final != prod * W_init (rel {}) - aborting.".format(_rel)
assert abs(_lr_max - _lr) <= LR_MAX_TOL, "schedule preflight: max lr {} != {} - aborting.".format(_lr_max, _lr)
del _p, _p_init, _opt, _sched, _ref
print("P2 schedule preflight ({:.1f}s): steps {} | prod(1 - wd*lr) = {!r} vs FRP {!r} (|diff| {:.2e} <= {}) | "
      "param rel dev {:.2e} <= {} | max lr {!r}: PASS".format(time.time() - t0, _steps, _prod, SHRINK_FINAL_EXACT,
                                                              SCHEDULE_PREFLIGHT["abs_diff_vs_frp"], SHRINK_TOL, _rel,
                                                              SCHED_PREFLIGHT_REL_TOL, _lr_max))

# --- P3 DataLoader length (built, NOT iterated) ---
for arch in ARCHS:
    _ld = make_loader(arch, ARMS[0], SEEDS[0])
    assert len(_ld) == EXPECTED_N_BATCHES_FINAL and EPOCHS * len(_ld) == EXPECTED_TOTAL_STEPS_FINAL, \
        "{}: len(loader) {} != {} - aborting.".format(arch, len(_ld), EXPECTED_N_BATCHES_FINAL)
    del _ld
assert len(train_rows) % BATCH_SIZE["resnet1d_wang"] == EXPECTED_LAST_BATCH_FINAL != 1
print("P3 DataLoader: len {} per epoch, {} total steps, last batch {} (not iterated): PASS".format(
    EXPECTED_N_BATCHES_FINAL, EXPECTED_TOTAL_STEPS_FINAL, EXPECTED_LAST_BATCH_FINAL))

# --- first write of this cell: OUT_DIR + the write-once preflight CSV (compared, never overwritten) ---
assert_no_v09("before the first write")
os.makedirs(OUT_DIR, exist_ok=True)
if os.path.exists(PREFLIGHT_CSV_PATH):
    _disk = pd.read_csv(PREFLIGHT_CSV_PATH, encoding="utf-8-sig", dtype=str)
    _mine = PREFLIGHT_DF.astype(str)
    assert list(_disk.columns) == list(_mine.columns) and _disk.reset_index(drop=True).equals(_mine.reset_index(drop=True)), \
        "v0.8_preflight_init_hashes.csv on disk differs from the recomputed preflight - aborting (never overwritten)."
    PREFLIGHT_CSV_SHA = sha256_of(PREFLIGHT_CSV_PATH)
    print("[exists] {} == recomputed preflight, row by row: PASS (not overwritten)".format(PREFLIGHT_CSV_PATH))
else:
    PREFLIGHT_CSV_SHA = save_csv_atomic(PREFLIGHT_DF, PREFLIGHT_CSV_PATH)
    print("[saved] " + PREFLIGHT_CSV_PATH)


# =====================================================================================
# RUN HELPERS, ATTEMPT LOG, train_run (v0.7 with the documented changes (9))
# =====================================================================================
# >>> verbatim from v0.7 (run naming)
def run_name(arch, arm, seed, lr):
    return "{}{}_{}_seed{}_lr{:g}".format(PREFIX, arch, arm, seed, lr)
# <<< verbatim from v0.7


def run_paths(arch, arm, seed, lr):
    """v0.8: two files per run (no probability file)."""
    base = os.path.join(OUT_DIR, run_name(arch, arm, seed, lr))
    return {"loss": base + "_train_loss.csv", "state": base + "_state_dict.pt"}


# >>> verbatim from v0.7 (partial + append-only attempt log)
def _partial(p):
    return p + ".partial"
# <<< verbatim from v0.7


# >>> verbatim from v0.7 (attempt log)
def read_attempt_log():
    entries = []
    if not os.path.exists(ATTEMPT_LOG_PATH):
        return entries
    with open(ATTEMPT_LOG_PATH, "r", encoding="utf-8") as f:
        raw = f.read()
    for i, line in enumerate(raw.splitlines()):
        if not line.strip():
            continue
        try:
            entries.append(json.loads(line))
        except json.JSONDecodeError:
            warn("attempt log line {} is not valid JSON (partial write?) - ignored: {!r}".format(i + 1, line[:120]))
    if raw and not raw.endswith("\n"):
        with open(ATTEMPT_LOG_PATH, "a", encoding="utf-8") as f:   # terminate a partial last line (append-only)
            f.write("\n")
    return entries


def log_event(run, attempt, event, **fields):
    entry = {"run": run, "attempt": attempt, "event": event, "time": datetime.now().astimezone().isoformat(), **fields}
    with open(ATTEMPT_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(plain(entry), ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())
    ATTEMPT_LOG.append(plain(entry))
    return entry
# <<< verbatim from v0.7


def loss_warn_flag(arch, final_loss):
    """WARN-only (LOSS_WARN_NOTE): no consequence, never a re-run criterion."""
    med = V07_MEDIAN_FINAL_LOSS[arch]
    return bool(final_loss > LOSS_WARN_HIGH_FACTOR * med or final_loss < LOSS_WARN_LOW_FACTOR * med)


def train_run(arch, arm, seed, lr, attempt):
    """Train 50 epochs on folds 1-9 and save the train-loss CSV and the final state_dict. NO evaluation.
    Adapted from v0.7 train_run; every change is listed as (9a)-(9h) in the header."""
    assert TRAINING_DETAILS_SHA is not None and os.path.isfile(TRAINING_DETAILS_PATH), \
        "training details must be written before any training run"
    name = run_name(arch, arm, seed, lr)
    paths = run_paths(arch, arm, seed, lr)
    bs = BATCH_SIZE[arch]
    assert len(train_rows) % bs != 1, "last training batch would have size 1 (BatchNorm) - aborting."
    assert len(train_rows) % bs == EXPECTED_LAST_BATCH_FINAL                                   # (9c)
    set_all_seeds(seed)
    model = ECGNet(arch, META[arm].shape[1])                                                   # (9b) CPU first
    init_shared, init_full = init_hashes(model)
    assert (init_shared, init_full) == (PREFLIGHT[(arch, arm, seed)]["init_shared_sha256"],
                                        PREFLIGHT[(arch, arm, seed)]["init_full_sha256"]), \
        "{}: init hashes differ from the preflight - aborting.".format(name)
    model = model.to(DEVICE)
    assert model.hidden[0].in_features == EXPECTED_HEAD_IN[arm], \
        "{}: head first-Linear in_features {} != {}".format(name, model.hidden[0].in_features, EXPECTED_HEAD_IN[arm])
    W0 = None
    if arm == "signal_plus_all":                                                                # (9e)
        W0 = model.hidden[0].weight.detach()[:, ZERO_COL_IDX].double().cpu().clone()
    loader = make_loader(arch, arm, seed)
    assert len(loader) == EXPECTED_N_BATCHES_FINAL and EPOCHS * len(loader) == EXPECTED_TOTAL_STEPS_FINAL, \
        "OneCycle total_steps {} != {}".format(EPOCHS * len(loader), EXPECTED_TOTAL_STEPS_FINAL)   # (9c)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=ADAMW_BETAS, eps=ADAMW_EPS, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EPOCHS * len(loader), **ONECYCLE)
    crit = nn.BCEWithLogitsLoss()
    loss_rows = []
    lr_prod, n_steps = 1.0, 0                                                                  # (9d) float64
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = torch.zeros((), device=DEVICE, dtype=torch.float64)
        n_seen = 0
        for xb, mb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            mb = mb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb, mb), yb)
            loss.backward()
            lr_now = float(opt.param_groups[0]["lr"])                                           # (9d)
            lr_prod *= (1.0 - WEIGHT_DECAY * lr_now)
            n_steps += 1
            opt.step()
            sched.step()
            loss_sum += loss.detach().double() * xb.shape[0]
            n_seen += xb.shape[0]
        mean_loss = float(loss_sum.item()) / n_seen
        assert math.isfinite(mean_loss), "{}: non-finite training loss at epoch {} - aborting.".format(name, epoch)
        loss_rows.append({"epoch": epoch, "mean_train_loss": mean_loss, "lr_end_of_epoch": sched.get_last_lr()[0]})
        if epoch % 10 == 0 or epoch == 1:
            print("    epoch {:2d}/{} | mean training loss {:.4f} | {:.0f}s".format(
                epoch, EPOCHS, mean_loss, time.time() - t0))
    torch.cuda.synchronize()
    wall = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024 ** 2
    # --- (i) integrity asserts (confirmations of the preflight) ---
    assert n_steps == EXPECTED_TOTAL_STEPS_FINAL, "{}: {} optimizer steps != {}".format(name, n_steps, EXPECTED_TOTAL_STEPS_FINAL)
    assert abs(lr_prod - SHRINK_FINAL_EXACT) <= SHRINK_TOL, \
        "{}: prod(1 - wd*lr) {!r} != FRP shrink factor {!r}".format(name, lr_prod, SHRINK_FINAL_EXACT)
    # --- (ii) non-finite parameters / buffers = "non-finite" under the crash rule (9f) ---
    sd = model.state_dict()
    bad = [k for k, t in sd.items() if t.is_floating_point() and not bool(torch.isfinite(t).all())]
    if bad:
        raise FloatingPointError("{}: non-finite state_dict tensor(s) {} - aborting.".format(name, bad[:10]))
    # --- (iii) WARN-only checks (9e, 9g) ---
    rel_dev = None
    if W0 is not None:
        W_final = model.hidden[0].weight.detach()[:, ZERO_COL_IDX].double().cpu()
        ref = lr_prod * W0
        rel_dev = float((W_final - ref).abs().max() / ref.abs().max())
        if rel_dev > ZERO_COL_REL_DEV_WARN:
            warn("{}: zero-column rel_dev {:.3e} > {} (WARN-only; no consequence)".format(name, rel_dev, ZERO_COL_REL_DEV_WARN))
    final_loss = loss_rows[-1]["mean_train_loss"]
    lw = loss_warn_flag(arch, final_loss)
    if lw:
        warn("{}: final-epoch train loss {:.4f} outside [{}x, {}x] the v0.7 {} median {} (WARN-only; never a re-run "
             "criterion)".format(name, final_loss, LOSS_WARN_LOW_FACTOR, LOSS_WARN_HIGH_FACTOR, arch, V07_MEDIAN_FINAL_LOSS[arch]))
    # --- (iv) loss CSV, (v) state_dict LAST (9h) ---
    pd.DataFrame(loss_rows).to_csv(_partial(paths["loss"]), index=False, encoding="utf-8-sig")
    os.replace(_partial(paths["loss"]), paths["loss"])
    torch.save(sd, _partial(paths["state"]))
    os.replace(_partial(paths["state"]), paths["state"])
    files_sha = {"loss": sha256_of(paths["loss"]), "state": sha256_of(paths["state"])}
    # --- (vi) log "completed" ---
    info = {"files_sha256": files_sha, "init_shared_sha256": init_shared, "init_full_sha256": init_full,
            "lr_product": lr_prod, "n_steps": n_steps, "zero_col_rel_dev": rel_dev,
            "final_epoch_train_loss": final_loss, "loss_warn": lw,
            "wall_time_s": round(wall, 1), "peak_vram_mb": round(peak_mb, 1)}
    log_event(name, attempt, "completed", **info)
    del model, opt, sched, loader, sd
    torch.cuda.empty_cache()
    return info


# =====================================================================================
# TRAINING (18 runs) with the crash / re-run rule
# =====================================================================================
hdr("TRAINING (18 runs, lr {}, folds 1-9, NO evaluation)".format(SELECTED_LR))
ATTEMPT_LOG = read_attempt_log()
TRAINED_THIS_EXECUTION = []
RECOVERED_NO_INIT_HASHES = []
RUN_INFO = {}
for i_run, (arch, arm, seed) in enumerate(RUN_ORDER, start=1):
    lr = SELECTED_LR[arch]
    name = run_name(arch, arm, seed, lr)
    paths = run_paths(arch, arm, seed, lr)
    entries = [e for e in ATTEMPT_LOG if e.get("run") == name]
    n_started = sum(e.get("event") == "started" for e in entries)
    if os.path.isfile(paths["state"]):
        # COMPLETE: the final state_dict exists -> never re-trained
        done = [e for e in entries if e.get("event") in ("completed", "completed_recovered")]
        if done:
            info = done[-1]
            for kind, p in paths.items():
                assert os.path.isfile(p) and sha256_of(p) == info["files_sha256"][kind], \
                    "{}: {} file differs from its logged sha256 - aborting (files must not change).".format(name, kind)
            if info["event"] == "completed":
                assert (info["init_shared_sha256"], info["init_full_sha256"]) == \
                    (PREFLIGHT[(arch, arm, seed)]["init_shared_sha256"], PREFLIGHT[(arch, arm, seed)]["init_full_sha256"]), \
                    "{}: logged init hashes differ from the preflight - aborting.".format(name)
            else:
                RECOVERED_NO_INIT_HASHES.append(name)
            print("  [{:2d}/18] {} -> COMPLETE ({}; hashes verified; not re-trained)".format(i_run, name, info["event"]))
        else:
            assert os.path.isfile(paths["loss"]), \
                "{}: state_dict exists but the train-loss CSV is missing and the log has no 'completed' entry - aborting.".format(name)
            _fl = float(pd.read_csv(paths["loss"], encoding="utf-8-sig")["mean_train_loss"].iloc[-1])
            info = log_event(name, n_started, "completed_recovered",
                             files_sha256={kind: sha256_of(p) for kind, p in paths.items()},
                             init_shared_sha256=None, init_full_sha256=None, lr_product=None, n_steps=None,
                             zero_col_rel_dev=None, final_epoch_train_loss=_fl, loss_warn=loss_warn_flag(arch, _fl),
                             wall_time_s=None, peak_vram_mb=None)
            RECOVERED_NO_INIT_HASHES.append(name)
            warn("{}: state_dict exists but the log has no 'completed' entry -> accepted as complete "
                 "('completed_recovered'; init / shrink fields null)".format(name))
    else:
        assert_no_v09("before training " + name)
        if n_started >= MAX_ATTEMPTS:
            raise RuntimeError("{}: {} attempts already started and no final state_dict exists -> STOP and report "
                               "(crash rule: at most {} attempts; stop and report; no setting may be changed except "
                               "through a written, externally timestamped amendment made before v0.9).".format(
                                   name, n_started, MAX_ATTEMPTS))
        stale = [p for p in list(paths.values()) + [_partial(q) for q in paths.values()] if os.path.exists(p)]
        if stale:
            for p in stale:
                os.remove(p)
            log_event(name, n_started, "deleted_partial_files", files=[os.path.basename(p) for p in stale])
            note("{}: deleted leftover files of the previous attempt: {}".format(name, [os.path.basename(p) for p in stale]))
        attempt = n_started + 1
        print("  [{:2d}/18] training {} | {} | seed {} | lr {:g} | attempt {} | {} epochs | batch {}".format(
            i_run, arch, arm, seed, lr, attempt, EPOCHS, BATCH_SIZE[arch]))
        log_event(name, attempt, "started")
        try:
            info = train_run(arch, arm, seed, lr, attempt)
        except BaseException as exc:
            log_event(name, attempt, "failed", error="{}: {}".format(type(exc).__name__, exc),
                      state_dict_exists=os.path.isfile(paths["state"]))
            print("  {} attempt {} FAILED -> logged; {}".format(
                name, attempt, "re-run the cell to start attempt {}".format(attempt + 1) if attempt < MAX_ATTEMPTS
                else "STOP and report (no further attempts allowed)"))
            raise
        TRAINED_THIS_EXECUTION.append(name)
        print("  -> final-epoch train loss {:.4f} | prod {:.10f} | zero-col rel_dev {} | {:.0f}s | peak VRAM {:.0f} MB".format(
            info["final_epoch_train_loss"], info["lr_product"],
            "n/a" if info["zero_col_rel_dev"] is None else "{:.2e}".format(info["zero_col_rel_dev"]),
            info["wall_time_s"], info["peak_vram_mb"]))
    RUN_INFO[(arch, arm, seed)] = {"name": name, "paths": paths, "info": info}
assert len(RUN_INFO) == 18 and all(os.path.isfile(RUN_INFO[k]["paths"]["state"]) for k in RUN_INFO)
print("all 18 runs complete ({} trained in this execution).".format(len(TRAINED_THIS_EXECUTION)))
del X
gc.collect()


# =====================================================================================
# SUMMARY TABLE (no metric columns) + MANIFEST
# =====================================================================================
hdr("TRAINING SUMMARY AND MANIFEST")
assert prereg_snapshot() == PREREG_SNAPSHOT_START, "PREREG_DIR CHANGED during v0.8 - aborting (v0.8 must never write there)."
PREREG_UNCHANGED = True
assert "wfdb" not in sys.modules and not _WFDB_AT_START, "wfdb was imported - aborting."
assert_no_v09("end of training")
ATTEMPT_LOG = read_attempt_log()
summary_rows, attempt_summary = [], {}
for arch, arm, seed in RUN_ORDER:
    r = RUN_INFO[(arch, arm, seed)]
    nm, inf = r["name"], r["info"]
    ev = [e for e in ATTEMPT_LOG if e.get("run") == nm]
    attempt_summary[nm] = {"started": sum(e["event"] == "started" for e in ev),
                           "failed": sum(e["event"] == "failed" for e in ev),
                           "completed": sum(e["event"] in ("completed", "completed_recovered") for e in ev),
                           "recovered": sum(e["event"] == "completed_recovered" for e in ev),
                           "deleted_partial_files": sum(e["event"] == "deleted_partial_files" for e in ev)}
    summary_rows.append({"arch": arch, "arm": arm, "seed": seed, "lr": SELECTED_LR[arch],
                         "attempts": attempt_summary[nm]["started"],
                         "final_epoch_train_loss": inf.get("final_epoch_train_loss"), "loss_warn": inf.get("loss_warn"),
                         "wall_time_s": inf.get("wall_time_s"), "peak_vram_mb": inf.get("peak_vram_mb"),
                         "init_shared_sha256": inf.get("init_shared_sha256"), "init_full_sha256": inf.get("init_full_sha256"),
                         "lr_product": inf.get("lr_product"), "n_steps": inf.get("n_steps"),
                         "zero_col_rel_dev": inf.get("zero_col_rel_dev"),
                         "state_dict_sha256": inf["files_sha256"]["state"], "train_loss_sha256": inf["files_sha256"]["loss"]})
summary_df = pd.DataFrame(summary_rows)
SUMMARY_CSV_PATH = os.path.join(OUT_DIR, PREFIX + "training_summary.csv")
save_csv_atomic(summary_df, SUMMARY_CSV_PATH)
print("[saved] " + SUMMARY_CSV_PATH)
print(summary_df[["arch", "arm", "seed", "attempts", "final_epoch_train_loss", "loss_warn", "wall_time_s", "peak_vram_mb",
                  "n_steps", "zero_col_rel_dev"]].to_string(index=False))
if RECOVERED_NO_INIT_HASHES:
    print("runs without logged init hashes (completed_recovered; init check skipped): {}".format(RECOVERED_NO_INIT_HASHES))
GPU_WALL_TOTAL = float(pd.to_numeric(summary_df["wall_time_s"], errors="coerce").sum(min_count=1))

OUTPUT_FILES = [RUN_INFO[k]["paths"][kind] for k in RUN_ORDER for kind in ("loss", "state")]
assert len(OUTPUT_FILES) == 36
OUTPUT_FILES += [PREFLIGHT_CSV_PATH, ATTEMPT_LOG_PATH, SUMMARY_CSV_PATH]
outputs = [{"file": p, "bytes": os.path.getsize(p), "sha256": sha256_of(p)} for p in OUTPUT_FILES]
for k in RUN_ORDER:
    for kind in ("loss", "state"):
        assert sha256_of(RUN_INFO[k]["paths"][kind]) == RUN_INFO[k]["info"]["files_sha256"][kind]

manifest = {
    "code_version": CODE_VERSION,
    "status": "FINAL TRAINING (folds 1-9), no evaluation",
    "run_started": RUN_STARTED,
    "run_finished": datetime.now().astimezone().isoformat(),
    "versions": VERSIONS, "gpu": GPU_NAME, "cudnn": CUDNN_VERSION,
    "parent_documents": {k: {"self_sha256": DOC_SELF[k], "file_sha256": DOC_FILE[k]} for k in DOC_PATHS},
    "ots_files": OTS_FILES, "ots_note": OTS_NOTE,
    "earlier_manifests_sha256": {"v0.3_manifest.json (from final_run_protocol, not opened)": V03_MAN_SHA_FROM_FRP,
                                 "v0.4_manifest.json": V04_MAN_SHA, "v0.5_manifest.json": V05_MAN_SHA,
                                 "v0.6_manifest.json": V06_MAN_SHA, "v0.7_manifest.json": V07_MAN_SHA},
    "input_files_sha256": INPUT_HASHES,
    "v0.4_signal_cache_sha256": CACHE_HASHES,
    "reference_implementation_sha256": REFERENCE_CODE_SHA256, "reference_implementation_exec_in_v0.8": False,
    "selected_lr": SELECTED_LR,
    "recipe_from_dev_protocol": {"epochs": EPOCHS, "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE, "crop": CROP},
    "v0.8_overrides_applied": True,
    "v0.8_overrides": V08_OVERRIDES,
    "stage_plan_v0.8": STAGE_PLAN_V08,
    "crash_and_rerun_rule_v0.8": CRASH_RULE_V08,
    "fold10_access": FOLD10_ACCESS_LINE,
    "locked_quantity_checks": LOCKED_CHECKS,
    "head_first_linear_in_features_v0.8": EXPECTED_HEAD_IN,
    "init_hash_recipe": INIT_HASH_RECIPE,
    "preflight_init_hashes": PREFLIGHT_DF.to_dict(orient="records"),
    "schedule_preflight": SCHEDULE_PREFLIGHT,
    "runs": summary_df.astype(object).where(pd.notna(summary_df), None).to_dict(orient="records"),
    "attempt_log_summary": attempt_summary,
    "attempt_log_entries": ATTEMPT_LOG,
    "trained_this_execution": TRAINED_THIS_EXECUTION,
    "completed_recovered_without_init_hashes": RECOVERED_NO_INIT_HASHES,
    "gpu_wall_time_total_s": GPU_WALL_TOTAL,
    "warn_only_notes": [LOSS_WARN_NOTE, ZERO_COL_WARN_NOTE],
    "loss_warn_reference_v0.7_median_final_loss": V07_MEDIAN_FINAL_LOSS,
    "warnings": WARNINGS,
    "flags": {"evaluation_performed": False, "metric_computed": False, "fold10_opened": False,
              "wfdb_imported": False, "prereg_dir_unchanged": PREREG_UNCHANGED,
              "files_read_from_outputs_v0.7": ["v0.7_manifest.json"], "v0.9_started": False},
    "outputs": outputs,
}
MANIFEST_PATH = os.path.join(OUT_DIR, PREFIX + "manifest.json")
with open(_partial(MANIFEST_PATH), "w", encoding="utf-8") as f:
    json.dump(plain(manifest), f, ensure_ascii=False, indent=2)
os.replace(_partial(MANIFEST_PATH), MANIFEST_PATH)
for o in outputs:
    print("  {}  {}".format(o["sha256"], o["file"]))
print("[saved] " + MANIFEST_PATH)
print("manifest sha256:", sha256_of(MANIFEST_PATH))


# =====================================================================================
# FINAL SUMMARY
# =====================================================================================
hdr("SUMMARY - " + CODE_VERSION)
for k in DOC_PATHS:
    print("  {:18s} self {} | file {} | .ots == v0.7 manifest: PASS".format(k, DOC_SELF[k], DOC_FILE[k]))
print("environment                : versions == v0.7 (torch {}), cuDNN {}, GPU {}".format(VERSIONS["torch"], CUDNN_VERSION, GPU_NAME))
print("locked quantities (folds 1-9) == final_run_protocol: encoder {} | meta dims {} | head widths {} | age scaler max "
      "diff {} | lead norm max diff {}".format(WIDTHS19, LOCKED_CHECKS["meta_dim"], EXPECTED_HEAD_IN, AGE_SCALER_DIFFS,
                                              LEAD_NORM_MAX_ABS_DIFF))
print("schedule preflight         : prod {!r} == FRP {!r}".format(SCHEDULE_PREFLIGHT["lr_product"], SHRINK_FINAL_EXACT))
print("runs: 18 complete | attempts per run: {} | trained in this execution: {}".format(
    {nm: v["started"] for nm, v in attempt_summary.items() if v["started"] != 1} or "all 1",
    len(TRAINED_THIS_EXECUTION)))
print("\nwarnings ({}):".format(len(WARNINGS)))
for w in WARNINGS:
    print("  - " + w)
print("total GPU wall time, 18 runs: {:.0f} s ({:.2f} h)".format(GPU_WALL_TOTAL, GPU_WALL_TOTAL / 3600.0)
      if math.isfinite(GPU_WALL_TOTAL) else "total GPU wall time: n/a (recovered runs without timing)")
assert prereg_snapshot() == PREREG_SNAPSHOT_START, "PREREG_DIR CHANGED during v0.8 - aborting."
assert "wfdb" not in sys.modules, "wfdb was imported - aborting."
assert_no_v09("end of cell")
print("\nv0.8: 18 runs trained on folds 1-9. No evaluation of any kind. Fold 10 not opened. PREREG_DIR unchanged.")



STEP 0 (part 1) - INTEGRITY
PREREG_DIR snapshot: 10 files
  babf101c1d69b322daef6907175fdc4c17b5a53755a68f13a998eefa0148dc2d  dev_protocol.json
  70c0ddb918108467cea97bddea0893ffa879aeeece8c941ea4fbdb46343a7a98  dev_protocol.json.ots
  e1a5260e72d2e8b0d27dc709639468b7f9a48a7c8e2f8af2d663e105dbd329d4  final_run_protocol.json
  585ea85d2d99dcb8c5060303def8dd8c20a5e72fa7f7df74b7d61fe66905719b  final_run_protocol.json.ots
  358d783b37bba6fcf61237a43817c1fd8594fe192f8ecdfca2c25b871a21415b  lr_selection.json
  7819486a0d2afe5cec2351fd71012d5c30512b6cf23c70acf4ae447419fa2463  lr_selection.json.ots
  11dd446272654fc80d65b2fdd40259fa674bbb419a44fde84e589b6e297cde32  pre_registration.json
  bb0c254734c52588bdfa5d4c1c4fa574eaa860f628c3f5c6b5402502acfba428  pre_registration.json.ots
  6480c191aeaed9ff1e6b1da5caa0c1e46908d453d30106a32beb7cf83678be11  v0.5_training_details.json
  497b7246a751a2d555aaee23f7c1b3ccf93b215893e13b3da82b7efdd59526d5  v0.5_training_details.json.ots
pre_registration   self